# <center>Harness Engineering 专题课第六节课 · FF-OpenHermes 源码解析</center>

&emsp;&emsp;这门课是 `Hermes Agent` 系列的源码实战落地篇，分两部分走。**第一部分「项目实战导览」** 带你把真实的 `FuFan-OpenHermes` 项目在本地跑起来——部署后端与前端、看懂它的界面和功能、理清前后端技术栈与整体架构。**第二部分「核心机制源码复现」** 再回到源码深处，用最小的 Python 把它的核心进化机制逐个亲手实现一遍。在前面的理论部分我们已经讲透了一个 Agent 为什么不该「用完即忘」、它该如何把经验沉淀成技能、又如何在复用中持续打磨自己；这门课要做的，就是把那套理论先跑通、再写透。

&emsp;&emsp;落地之后的它，不再是「问一句答一句」的助手——用过几次工具后，它会自己把工作流提炼成一份 `SKILL.md`；复用时反思「上次哪里做得不够好」并改进它；还会每隔几轮对话由后台定期触发一次复盘，把值得长期记住的东西写进自己的记忆文件，甚至定期请大模型帮自己「整理房间」，把杂乱重复的记忆重写得干净有条理。

&emsp;&emsp;这套能力，在 `FuFan-OpenHermes` 这个真实项目里被拆成了一组互相配合的机制。第一部分你会先以「用户视角」把它跑起来、摸清每个界面背后是什么能力、它的技术栈和架构长什么样；第二部分我们换成「实现者视角」，逐个机制贴上 `file:line` 锚点对照，用最小的 Python 重新写一遍。当你能用自己的代码跑通「检测工具调用 → 大模型判定 → 写技能文件」这条链路，再回头看项目源码里那几百行，你会发现它只是把同一套逻辑做得更健壮而已。换句话说，我们追求的不是「看懂别人的代码」，而是「自己也能写出来」。

> 📌 **目标受众与前置要求**：本课面向有 Python 基础、用过 LLM API、想搞懂「一个 Agent 怎么自己写技能、自己改进技能、自己反思记忆」的开发者。技术上你需要会读 `async/await` 风格的代码、调用过任意一家大模型的 Chat API，**不需要** 读过 `LangChain` 源码。第一部分会带你完整部署一次真实项目（需要本地有 `Node.js` 环境）；第二部分的每一章代码都用临时目录和内联数据做到自包含，复制进 Notebook 就能跑，**不依赖** 第一部分的工程环境——两部分可以分开学。

> 📌 **学完本节你将带走 8 件产物**：① 用 Python 复现的三层记忆（`MiniMemoryOps` 读写 + `.bak` 备份）；② 一个把三层记忆 + 技能按五层拼成 `system_prompt` 的分层拼接器（含技能 Level 0 快照 / Level 1 全文两级加载）；③ 对 `AgentMiddleware` 钩子触发时机的实测理解；④ 一套可直接复用到任何项目的 `json_mode + PydanticOutputParser` 结构化输出范式；⑤ M2 技能自主生成的完整大模型调用链；⑥ M3「Actor 反思 + Curator 决策」双段强化的最小实现；⑦ HIL 危险命令拦截的 `asyncio.Future` 反向通道模式；⑧ 能跑 HQS 会话诊断并解读 5 类失败模式的能力。

> 📌 **学完不能做（诚实划界）**：本课复现的是进化机制的**骨架最小实现**，不是生产级实现——我们不深入前端 `React` 链路、不覆盖生产部署的 `Docker` 编排，也不展开技能 lint 校验、事件总线等支撑层（这些源码里都有，只是不在「自我进化」核心主线上）。完整的诚实划界见第 13.7 节。

> 📅 **时效性说明**：本课全部源码引用截止 2026 年 6 月，基于 `FuFan-OpenHermes` 项目当时的代码状态；运行环境为 `conda env fufan-openhermes`（Python 3.11.15），大模型统一使用 DeepSeek 官方端点。所有 `file:line` 引用都是真实可核对的——你可以在自己电脑上打开对应源码文件跳到那一行验证。课件里出现的具体版本号集中在本说明和 `requirements.txt` 里声明一次，正文不再重复散落。

# <center>第一部分：项目实战导览</center>

&emsp;&emsp;在正式进入源码之前，我们先把 `FuFan-OpenHermes` 这个真实项目跑起来，用「用户视角」把它摸清楚。第一部分共三章：第 1 章带你完成环境部署和项目启动，第 2 章带你查看所有功能界面建立全景认知，第 3 章把前后端技术栈和整体架构梳理清楚。这三章结束之后，你对「这个东西长什么样、能做什么、由什么搭成的」已经有了直感——第二部分再换成「实现者视角」，逐个机制贴上源码锚点用 Python 亲手复现，才不会迷失在细节里。

## <center>第1章：环境部署与项目启动</center>

&emsp;&emsp;本课第一部分带你把真实 `FuFan-OpenHermes` 项目跑起来；跑通后第二部分再用最小 Python 复现它的核心机制。这一章我们解决「能不能跑」的问题——环境配齐、后端启动、前端启动、发出第一条消息。只有亲眼看到 Agent 流式回复，后续的源码拆解才有了对照的「活的参照物」。

> **【关于本章代码块格式】**：以下环境部署命令以 code cell（`!` 前缀）形式给出，针对真实 `FuFan-OpenHermes` 项目，请在你本地项目目录的终端中执行（路径按本地结构调整）。创建 conda 环境、装依赖、跑测试可直接运行；但 `conda activate`、启动服务（`uvicorn` / `npm run dev`）等受 Jupyter 限制（子进程激活无效 / 阻塞挂起），请复制到独立终端执行。

### 1.1 环境依赖清单

&emsp;&emsp;在启动之前，先确认运行环境满足以下要求。这里是一份**部署装机清单**——只列启动必需的运行时版本、依赖安装入口和密钥；至于后端前端各用了哪些框架、为什么这么选，留到第 3 章「架构与技术栈」从架构视角系统展开，这里不展开技术细节。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>FuFan-OpenHermes 运行环境与依赖</font></p>
<div class="center">

| 类别 | 要求 | 说明 |
|------|------|------|
| Python | ≥3.10（建议 3.11）| 后端运行时 |
| Node.js | ≥18（建议 20 LTS）| 前端构建运行时 |
| 后端依赖 | `requirements.txt` 一键装 | `pip install -r requirements.txt` 自动装齐 `FastAPI` / `LangChain 1.x` 等（完整技术栈见第 3 章）|
| 前端依赖 | `package.json` 一键装 | `npm install` 自动装齐 `Next.js 14` 等 |
| 密钥（必填）| `OPENROUTER_API_KEY` | 单 key，走 OpenRouter 统一路由大模型 |
| 密钥（可选）| `TAVILY_API_KEY` | 不填则禁用 `search_web` 联网搜索 |
| 存储 | 纯本地文件系统 | 无数据库、无向量库，每 session 一个目录 |

</div>

&emsp;&emsp;关于密钥有一点要特别说明：本项目使用**单 key** `OPENROUTER_API_KEY`，走 OpenRouter 统一路由到各家大模型；`TAVILY_API_KEY` 是可选的，不填只是禁用联网搜索工具，其余功能不受影响。**没有 OpenAI Embedding key，没有双 key 配置**——这是项目的架构选择，它让部署极度简化。

### 1.2 后端启动

&emsp;&emsp;后端是一个标准的 `FastAPI` 应用，启动流程分五步。按顺序执行，每一步都确认正常再往下走。

&emsp;&emsp;<font color=red>**先确认工作目录**</font>：除「创建 conda 环境」外，下面的依赖安装、测试、启动命令都必须在项目的 `backend/` 子目录下执行（`requirements.txt`、`app.py` 都在那里）。在终端里先 `cd <你克隆项目的路径>/backend` 再跑这些命令；如果在 Jupyter 里运行，用 `%cd <项目路径>/backend` 魔法命令切换工作目录——注意 `!cd` 是在 `!` 子进程里切换、对 kernel 不生效，必须用 `%cd`。

**步骤一：创建并激活 conda 环境**

&emsp;&emsp;本课全程使用 conda 环境，与第二部分统一用 `fufan-openhermes` 环境。用 conda 新建一个独立环境隔离依赖，避免污染 base 或其他项目；激活之后，后续的 `pip install` 才会装进这个环境。

In [ ]:
# 创建独立 conda 环境（!conda create 可直接运行；若已有 fufan-openhermes 环境可跳过本行）
!conda create -n fufan-openhermes python=3.11 -y

# 注意：!conda activate 在 Notebook 的 ! 子进程里激活，不会改变当前 kernel 环境。
# 正确做法是在终端激活该环境后用它启动 Jupyter；下面一行仅作命令记录：
!conda activate fufan-openhermes

**步骤二：安装后端依赖**

&emsp;&emsp;依赖版本锁定在 `requirements.txt`，一条命令装齐。

In [ ]:
!pip install -r requirements.txt -q

**步骤三：配置环境变量**

&emsp;&emsp;从模板复制一份 `.env` 文件，然后填入你的 `OPENROUTER_API_KEY`。按本课 dotenv 规范，密钥走 `.env` 文件加载，禁止硬编码。`.env` 文件不要提交到 Git，建议在 `.gitignore` 里加上它。

In [ ]:
!cp .env.example .env
# 然后用编辑器打开 .env，填入：
# OPENROUTER_API_KEY=your_key_here
# TAVILY_API_KEY=your_tavily_key_here  （可选）

**步骤四：运行测试，验证环境**

&emsp;&emsp;项目自带完整的测试套件，全部跑通代表环境配置正确。如果有测试失败，通常是依赖版本不对，检查 Python 版本和 `pip install` 是否在正确的虚拟环境里执行。

In [ ]:
!pytest -v
# 期望结果：测试全部通过（all passed）

**步骤五：启动后端服务**

&emsp;&emsp;后端监听本地 `8003` 端口（项目默认值；如需修改可在 `.env` 里调 `PORT` 变量）。

In [ ]:
# 启动后端服务（阻塞式长进程，会一直占用本 cell —— 请在独立终端运行）
!uvicorn app:app --reload --host 127.0.0.1 --port 8003

&emsp;&emsp;看到终端输出 `Application startup complete.` 说明后端已就绪。可以访问 `http://127.0.0.1:8003/openapi.json` 看到 API 路由列表作为二次确认。

> **【踩坑预警】**：OpenRouter 在部分地区会屏蔽 `anthropic/openai/google` 系模型返回 403。默认模型 `deepseek/deepseek-chat-v3.1` 对此免疫，开箱即用。如果你的 OpenRouter 额度不足或受限，也可以把 `OPENROUTER_BASE_URL` 指向 DeepSeek 官方端点（`https://api.deepseek.com/v1`）——这正是第二部分源码复现采用的端点，两者都是 OpenAI 兼容格式，可以无缝切换。

### 1.3 前端启动

&emsp;&emsp;前端是一个 `Next.js` App Router 应用，启动同样三步。确保后端已经启动再执行前端启动，否则首次加载会看到 API 连接失败的提示。

&emsp;&emsp;<font color=red>**先确认工作目录**</font>：下面三条命令都必须在项目的 `frontend/` 子目录下执行（`package.json` 在那里）。终端里先 `cd <项目路径>/frontend`，Jupyter 里用 `%cd <项目路径>/frontend`（同样地，`!cd` 对 kernel 无效）。

**步骤一：安装前端依赖**

In [ ]:
!npm install

**步骤二：构建前端**

In [ ]:
!npm run build
# 期望：Compiled successfully（编译成功）

**步骤三：启动开发服务器**

In [ ]:
# 启动前端开发服务器（阻塞式长进程 —— 请在独立终端运行）
!npm run dev
# 前端监听 http://localhost:3000

&emsp;&emsp;浏览器打开 `http://localhost:3000`，看到聊天界面说明前端正常。

### 1.4 首条对话验证

&emsp;&emsp;前后端都起来了，用一条最简单的消息验证整条链路是通的。

**步骤一：新建会话并绑定工作目录**

&emsp;&emsp;在界面左侧点「新建会话」，系统会提示你选择一个本地目录作为这个 Agent 的工作区。选一个空目录（或新建一个测试目录），`workspace` 绑定成功后左侧文件树就会显示该目录内容。

**步骤二：发送第一条消息**

&emsp;&emsp;在输入框发送：「列一下当前工作目录的文件」。Agent 会调用 `terminal` 工具执行 `ls`，流式回复显示文件列表。如果你看到流式文字一字一字地出现，说明 `sse-starlette` SSE 流式链路完全正常。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165901068.png" width=50%></div>

## <center>第2章：功能全景与界面操作</center>

&emsp;&emsp;跑起来后，先建立「真实产品长什么样」的整体认知。第二部分每个机制都能回指本章某个界面——理解界面功能，源码机制才有了具体的锚点。`FuFan-OpenHermes` 前端有 5 个页面：`/`（聊天主界面）、`/memory`（记忆管理）、`/skills`（技能库）、`/distill`（技能蒸馏）、`/train`（技能训练），每个页面背后都对应着第二部分要复现的某个核心机制。

&emsp;&emsp;这一章不是功能手册，而是「功能 × 机制」的映射建立。我们逐个页面走一遍，重点看「这个界面让你操作的，背后是第二部分哪一章的代码」——有了这张映射，后续每当你在 Python 里复现某个机制，脑子里就能自然地浮出它在界面上是什么样子的。

### 2.1 整体布局

&emsp;&emsp;打开 `http://localhost:3000`，看到的是聊天主界面。整体是三栏式布局：左侧栏包含会话列表（历史对话切换）和工作目录文件树（实时显示绑定目录的文件变化）；中间是对话流区域，消息以气泡形式显示，Agent 的回复是流式输出；底部是输入框，支持 `Shift+Enter` 换行。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165910586.png" width=92%></div>

&emsp;&emsp;上图就是真实运行的聊天主界面：最左是工作台导航（`对话` / `记忆` / `技能`）和技能工坊（`蒸馏` / `训练`）——这正是我们说的 5 个页面入口，下方是绑定工作目录的文件树（图中 `ff_demo_ws` 里已出现 Agent 刚创建的 `todo.md`）；中间对话流里，那几个带 ✓ 的折叠卡片（`write_file` / `terminal` / `write_memory`）就是 Agent 这一轮调用的工具，下一节我们专门看它们。

&emsp;&emsp;工作目录绑定是 `FuFan-OpenHermes` 有别于普通聊天工具的关键设计：Agent 的所有文件操作（`read_file` / `write_file` / `terminal`）都在这个绑定目录里发生，会话结束后你可以直接打开目录查看 Agent 留下的产物。这是「让 Agent 真正做事」而不是「纸上谈兵」的前提。

### 2.2 对话系统与核心工具集

&emsp;&emsp;对话采用 `sse-starlette` 实现的 SSE（Server-Sent Events）流式输出——大模型的 token 生成一个推送一个，你能看到 Agent 逐字回复而不是等全部生成完再一次性展示。Agent 在对话中可以调用**一组工具**，覆盖文件操作、代码执行、网络访问和记忆读写等场景。这里我们不纠结工具的确切数量——它是开放的，项目会随需求继续新增（比如第二部分会遇到的跨会话检索工具 `session_search`）；下面只列出最核心、通用智能体最常用的那几个。

<p align="center"><font face="黑体" size=4>Agent 可调用的核心工具</font></p>
<div class="center">
<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>

| 工具名 | 用途 | 源码文件（`backend/tools/` 下）|
|--------|------|------|
| `terminal` | 在绑定工作目录里执行 shell 命令（ls / mkdir / git 等） | `terminal_tool.py` |
| `python_repl` | 执行 Python 代码片段，结果返回给 Agent | `python_repl_tool.py` |
| `fetch_url` | 抓取指定 URL 的网页内容 | `fetch_url_tool.py` |
| `search_web` | 调用 Tavily 联网搜索（需配置 `TAVILY_API_KEY`） | `search_tool.py` |
| `read_file` | 读取工作目录内的文件内容 | `file_tool.py` |
| `write_file` | 向工作目录内的文件写入内容 | `file_tool.py` |
| `read_memory` | 读取当前会话的 `MEMORY.md` 长期记忆 | `memory_tool.py` |
| `write_memory` | 向 `MEMORY.md` 写入新的记忆条目 | `memory_tool.py` |
| `read_skill` | 读取 `skills/` 目录下已生成的技能文档 | `skill_tool.py` |

</div>

&emsp;&emsp;这张表只列了最核心的工具，<font color=red>真正要记住的不是「一共有几个」，而是 `read_memory` / `write_memory` / `read_skill` 这三个工具——它们是自我进化机制的核心</font>，让 Agent 能在工具调用层面直接读写自己的记忆和技能，这是第二部分要深入拆解的重点。工具集本身是开放的：项目随时可以按需挂上新工具（第二部分我们就会给它加一个跨会话检索记忆的 `session_search`），所以不必纠结某个固定的工具总数。

### 2.3 `/memory` 页：三层记忆 + 诊断

&emsp;&emsp;点击顶部导航的 `Memory` 进入 `/memory` 页。这个页面直接暴露了 Agent 的「内存状态」，分三个区域：

&emsp;&emsp;第一个区域是 `SOUL.md`——Agent 的人格层，显示 Agent 的角色定位和回答风格。这个文件极少变化，通常由项目维护者手动配置。第二个区域是 `MEMORY.md`——嵌入了 `Monaco Editor`（VSCode 同款编辑器），你可以直接在浏览器里编辑这份长期记忆，右上角有「AI 优化」按钮，点击后大模型会对当前记忆做去重 / 分区重写并自动 `.bak` 备份（这是第二部分第 11 章的核心机制）。第三个区域是会话历史，列出当前 session 的对话摘要，每条会话旁有「诊断本次会话」按钮，点击后 LLM 对整段会话打 HQS 分数（这是第 12 章的机制）。

&emsp;&emsp;这个页面最直观地展示了「三层记忆」的样子：`SOUL.md`（人格层）→ `MEMORY.md`（长期记忆层）→ 会话历史（会话层，底层是 `SqliteSaver`）。第二部分第 5 章会把这三层用 Python 复现出来。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165914977.png" width=92%></div>

&emsp;&emsp;如上图，页面顶部三张卡片就是三层记忆的直观呈现——`SOUL.md`（人格层）、`MEMORY.md`（长期记忆层）、`会话历史`（会话层，标注 `SqliteSaver`），每张卡片还标了它当前占用的 token 量；下方是 `Monaco Editor`，可以直接在浏览器里编辑这份记忆并保存。

### 2.4 `/skills` 页：技能库 + 演进

&emsp;&emsp;点击顶部导航的 `Skills` 进入 `/skills` 页。这里显示 Agent 自主生成的所有技能文档（`SKILL.md` 格式），每个技能卡片显示技能名、描述和版本号。点击某个技能可以看到它的演进历史——从 `v1.0` 到 `v1.1`，每次强化的 diff 都有记录。

&emsp;&emsp;这个页面让你直接看到「自我进化」的产物：Agent 用了 3 次以上工具完成某个任务后，会在 `aafter_agent` 钩子里异步生成一个 `SKILL.md` 文档，下次遇到相似任务就能复用，复用时反思并强化它。这正是第二部分第 8、9 章要复现的 M2（技能生成）和 M3（技能强化）。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165917966.png" width=48%></div>

&emsp;&emsp;仔细看技能卡片右侧的来源徽章，这是这个页面最值得讲的细节：<font color=red>蓝色「你教的」表示这条技能是你在对话里明确要求存下的，绿色「它自学的」表示是 Agent 在后台复盘时自己沉淀的</font>。这条「双重来源（provenance）」的区分贯穿整个自我进化主线——它让技能库始终分得清「人类教的」和「机器自学的」，也是后面技能库自动治理的安全边界：Agent 整理、合并技能时只会动它自己学的，绝不碰你手工教它的。

### 2.5 `/distill` 页：手动蒸馏技能

&emsp;&emsp;点击左侧「技能工坊」下的 `蒸馏` 进入 `/distill` 页。如果说 `/skills` 页看到的技能是 Agent **自动**沉淀的，那么这个页面给你一个**手动**触发技能生成的入口：把一段有价值的对话粘贴进文本框（或直接点「用本会话历史」让它读取当前会话），再点「蒸馏」，大模型就会判断这段内容值不值得提炼成技能，值得的话自动命名并写入技能库。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165919937.png" width=92%></div>

&emsp;&emsp;它背后正是第二部分第 8 章要复现的 M2 技能生成机制——同样的「判定 → 生成」两步，区别只在触发方式：`/skills` 页的技能由后台钩子自动触发，而 `/distill` 页让你随时手动触发同一套逻辑，方便你把一段满意的对话立刻固化成技能。

### 2.6 `/train` 页：多轮训练强化技能

&emsp;&emsp;点击「技能工坊」下的 `训练` 进入 `/train` 页。技能第一次生成时往往并不完美，这个页面让你**定向打磨**某个技能：选一个技能、给它出几道任务、写明评判标准，点「开始训练」，系统就会多轮迭代——每轮让技能去做题（Actor）、对照标准自评打分（Evaluator）、再根据反思给技能打补丁（Curator），右侧实时显示 `passRate` 通过率曲线一轮轮爬升。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165922266.png" width=92%></div>

&emsp;&emsp;这正是第二部分第 9 章要复现的 M3 技能强化机制（Verbal RL，用自然语言反思代替梯度更新）。可以这样区分三个技能页：`/skills` 看技能「有没有」、`/distill` 解决「从无到有」、`/train` 解决「从有到优」。

### 2.7 其余高级特性概览

&emsp;&emsp;除了基础对话，`FuFan-OpenHermes` 还有几项高级特性，都在界面上有对应的触发入口：

<p align="center"><font face="黑体" size=4>高级特性与界面触发方式</font></p>
<div class="center">
<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>

| 特性 | 复现章节 | 触发方式 | 界面表现 |
|------|------|---------|---------|
| HIL 危险命令确认 | 第 10 章 | Agent 执行 `rm` / `mv` 等危险命令时自动弹出 | 对话流中出现确认弹窗，等待用户点允许/拒绝 |
| AI 优化 MEMORY | 第 11 章 | 点击 `/memory` 页的「AI 优化」按钮 | `MEMORY.md` 被大模型重写，旧版自动 `.bak` |
| 模型切换 + Token 条 | 第 13 章 | 顶部下拉切换模型 | 可切换 4 个预设模型；实时显示本次 Token 消耗 |
| HQS 会话诊断 | 第 12 章 | 点击会话旁「诊断」按钮 | 弹出诊断报告：0-10 分 + 5 类失败模式分析 |

</div>

&emsp;&emsp;**诚实边界声明**：本项目聚焦「记忆 + 技能进化 + 安全/诊断」这条自我进化主线，**不含** RAG 检索增强、浏览器自动化、Canvas 实时渲染、可视化技能商店这类外围能力——它的重心全在三层记忆和技能演进上。如果你需要上述功能，那是另外的项目方向，本课不覆盖。

## <center>第3章：架构与前后端技术栈</center>

&emsp;&emsp;从「会用」过渡到「看懂结构」——本章是第二部分的源码总图。有了界面认知（第 2 章），现在我们看它由什么搭成、各层是怎么协作的，让你在进入第二部分源码之前，先在脑子里建起一张「前后端整体架构图」。有了这张图，后面每当我们说「这段代码在 Agent runtime 层」或者「这个工具调用走的是工具层」，你就能立刻定位到它在全图中的位置。

&emsp;&emsp;本章信息量较大，但结构清晰——技术栈表 → 整体架构图 → 仓库目录导读 → 一条消息的生命周期。后三项都是给第二部分「按图索骥」用的参考，不要求一次记住，看懂结构即可，后面需要时随时回来查。

### 3.1 前后端技术栈表

&emsp;&emsp;第 1 章 1.1 的装机清单只告诉你「要装什么版本」；这一节我们换成架构视角，把每一层各用了什么、为什么这么选，完整摊开。本项目技术选型极度克制，每一层只选了能解决问题的最小必要工具组合——下面这张表可以当作整个第二部分的「技术索引」。

<p align="center"><font face="黑体" size=4>FuFan-OpenHermes 前后端技术栈</font></p>
<div class="center">
<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>

| 层 | 技术 | 说明 |
|----|------|------|
| 后端语言 | Python 3.10+ | 建议 3.11 |
| 后端框架 | `FastAPI` + `Pydantic v2` + `sse-starlette` | REST API + 流式推送 |
| Agent 运行时 | `LangChain 1.x`（`create_agent`）+ `LangGraph`（`AsyncSqliteSaver`）| Agent 编排 + 会话持久化 |
| 前端框架 | `Next.js 14` App Router / `TypeScript` strict | 5 页路由 |
| 前端状态 | `Zustand` | 全局状态管理 |
| 前端编辑器 | `Monaco Editor` | VSCode 同款，用于 MEMORY.md 在线编辑 |
| 前端样式 | `Tailwind v3` + `react-markdown` | 样式 + Markdown 渲染 |
| 默认 LLM | OpenRouter `deepseek/deepseek-chat-v3.1` | 4 个预设模型可切换 |
| LLM 路由 | OpenRouter（单 key `OPENROUTER_API_KEY`）| 统一路由多家 LLM |
| 持久化 | 纯本地文件系统 `sessions/{sid}/` | SOUL.md + MEMORY.md + state.db + skills/ + diagnostics.json |
| 事件流 | `sse-starlette`（主流）+ `/api/events` 旁路 | Agent 思考过程实时推送 |

</div>

&emsp;&emsp;注意几个选型里的「为什么」：用 `AsyncSqliteSaver` 而不是 PostgreSQL，是因为「每个 session 独立一个 sqlite 文件」天然隔离、零运维成本；用纯文件系统存记忆而不是向量库，是因为记忆需要可读可编辑——这些选择在第二部分各章都会遇到，届时结合源码就能更深地理解它们。

### 3.2 五层架构分解

&emsp;&emsp;从技术栈看「有什么」，从架构图看「怎么连」。本项目的架构可以分成五层，从前端到存储，每一层的职责都很清晰。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165926571.png" width=50%></div>

&emsp;&emsp;这五层里，第二部分会重点实现第三层（Agent Runtime）和第四层（工具层）的核心机制——前端和 API 层不在本课覆盖范围内，但有了这张全图，你清楚知道「我在哪里」。

### 3.3 仓库目录导读

&emsp;&emsp;打开项目仓库，后端目录结构如下：`agent/` 目录是 Agent 运行时核心，包含 `memory_ops.py`（三层记忆读写）、`middleware.py`（钩子触发器）、`memory_optimizer.py`（AI 优化记忆）、`diagnostics.py`（HQS 诊断）、`permission_registry.py`（HIL 反向通道）；`skill_engine/` 目录是技能进化引擎，包含 `generator.py`（M2 技能生成）、`evolver.py`（M3 技能 Verbal RL 训练）、`curator.py`（技能库治理）、`background_review.py`（后台复盘子 agent）；`api/` 目录是 FastAPI 路由；`tools/` 目录是各个工具的实现。前端目录：`app/` 下是 5 个页面路由（聊天 / memory / skills / distill / train），`components/` 是 chat / memory / skills 组件，`stores/` 是 `Zustand` 状态管理，`hooks/` 里的 `useSseStream` 是 SSE 流式消费钩子。

&emsp;&emsp;这份目录导读是第二部分的「源码索引」——每当我们说「源码锚点 `agent/memory_ops.py:16`」，你回到这份导读就能快速定位到「在 `agent/` 目录下」。

### 3.4 一条消息的生命周期

&emsp;&emsp;最后用一张流程图，把「用户发一条消息到 Agent 回复完成」这整条链路串起来。这是第二部分所有机制的「运行时容器」——M2、M3、HIL、自省等都是这条生命周期某个节点上的旁路逻辑。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165933023.png" width=50%></div>

&emsp;&emsp;第一部分到这里结束。你已经把 `FuFan-OpenHermes` 跑起来了，查看过所有界面，理清了前后端架构。现在切换到「实现者视角」——第二部分我们用 Python 逐个机制亲手复现它。

# <center>第二部分：核心机制源码复现</center>

&emsp;&emsp;第一部分你已经以「用户视角」把 `FuFan-OpenHermes` 跑起来、摸清了它的界面和架构。第二部分切换到「实现者视角」：我们不再关心界面，只关心背后跑的机制——逐章取出一个机制，贴上 `file:line` 锚点，用最小的 Python 亲手复现它。每一章的代码都自包含（临时目录 + 内联数据），复制进 Notebook 就能跑，不依赖第一部分的工程环境。

## <center>第4章：源码地图——进化机制的模块总图</center>

&emsp;&emsp;进入实现者视角的第一步，这一章我们只做一件事：把自我进化的闭环**快速锚定到 `FuFan-OpenHermes` 的源码模块上**，给接下来八章的源码拆解立一张地图。否则后面每一章都在实现一个机制，你很容易迷失在细节里，忘了它们到底拼成了一张什么样的图。至于「一个 Agent 为什么要自我进化、它的闭环长什么样」这套理论，更早的理论课已经讲过，这里不再展开。

### 4.1 把闭环锚定到源码：四步循环对应哪些模块

&emsp;&emsp;先用一张图把这条主线在脑子里建立起来。自我进化的核心，是一个 `Solve → Document → Improve → Repeat` 的四步闭环。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165939931.png" width=50%></div>

&emsp;&emsp;这个闭环就是：解决任务（Solve）→ 把有用经验写成技能文档（Document）→ 复用时反思改进（Improve）→ 带着更强的能力进入下一轮（Repeat）。它在 `FuFan-OpenHermes` 里不是一句口号，而是一组真实模块——Document 这一步对应 `generator.py`（生成技能），Improve 这一步对应 `evolver.py`（训练强化技能），我们后面会逐行对照。需要提醒一句：**「自我进化」是我们对项目机制的提炼命名，不是业界标准术语，源码里找不到这个词**，但它精准对应了上面这两个模块的真实行为。

### 4.2 这门课的主线：第二部分各章按进化依赖顺序排列

&emsp;&emsp;理解了闭环，我们就能看懂这门课的章节编排逻辑。它不是随便排的，而是严格按照「进化机制的依赖顺序」往下走的——后面的机制要用到前面机制的产物，所以必须先打地基。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>主线各章与进化闭环的对应关系</font></p>
<div class="center">

| 章节 | 机制 | 在进化闭环中的角色 |
|------|------|-------------------|
| 第5章 | 三层记忆 | 进化的地基——它能分层记住什么 |
| 第6章 | System Prompt 分层拼接 | 进化的地基——记忆怎么拼成提示词喂给模型 |
| 第7章 | 中间件钩子 | 进化的触发器——「用过工具就反思」的入口 |
| 第8章 | 技能自主生成（M2）| Document——把工作流写成技能 |
| 第9章 | 技能自主强化（M3）| Improve——复用时改进技能 |
| 第10章 | HIL 危险命令拦截 | 自主但可控——进化的前提是不失控 |
| 第11章 | 自省 + AI 优化记忆 | 记忆的自我管理¹——整理自己的房间 |
| 第12章 | HQS 会话诊断 | 自我评估¹——知道自己哪里做得不好 |
| 第13章 | 闭环回顾 + 生产化补充 | 把所有机制装配回完整闭环 |

</div>

> &emsp;¹「自我管理」「自我评估」和本课其他几个讲师提炼命名（「自我进化」「自我迭代」「三层记忆」）一样，都不是业界标准术语，在 `FuFan-OpenHermes` 真实源码里找不到对应词——但它们各自精准对应一个真实模块：「自我管理」对应 `memory_optimizer.py`，「自我评估」对应 `diagnostics.py`。

&emsp;&emsp;从这张表你能看出一条清晰的递进：前两章打地基（记忆 + 触发器），第 8、9 章是这门课的双核（生成 + 强化，对应进化闭环最关键的 Document 和 Improve），第 10 到 12 章是让进化「安全、可管理、可评估」的三道保险，最后一章把它们串成一个完整的生命周期。你会在后面章节标题里看到 `M2`、`M3`、`M4-A` 这类代号——它们是项目给各功能模块起的内部编号：`M2` 指技能生成、`M3` 指技能强化，`M4-A`/`M4-C`/`M4-D`/`M4-E` 分别是 HIL、AI 优化记忆、模型切换、HQS 诊断这几项高级特性。记不住没关系，每章标题都会标清楚，这里只需知道它们是同一套编号体系即可。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165943133.png" width=50%></div>

&emsp;&emsp;动手之前，先把第二部分的运行前提一次说清，后面各章就不再重复：每一章的代码都做到自包含——用临时目录和内联数据，复制进 Notebook 就能跑，不依赖第一部分的工程环境。其中第 5、6 章是纯文件操作、不调用大模型，没有任何 key 也能跑通；从第 7 章起才真实调用 DeepSeek，需要在 `~/.claude/.env` 里配好 `DEEPSEEK_API_KEY`（运行环境统一用 conda `fufan-openhermes` 环境、Python 3.11）。还有两个会被反复复用的「公共资产」：第 7 章定义的大模型工厂 `build_mini_llm`、第 8 章封装的结构化输出范式 `structured_call`——后面章节用到时我们直接引用，不再重复讲解。这里有一个**运行前提**要先讲清：第 7 章及之后的章节要在同一个 Jupyter kernel 会话里按顺序运行；如果中途重启了 kernel，需要从第 7 章起把这两个公共资产的 cell 重新跑一遍，否则后面会报 `NameError`（提示 `build_mini_llm` 未定义）。还要注意一个 key 的区别：第一部分项目默认走 OpenRouter（用 `OPENROUTER_API_KEY`），第二部分复现统一改用 DeepSeek 官方端点（用 `DEEPSEEK_API_KEY`），两个 key 不要混用。还有一点要先说明：第二部分有不少 cell 在顶层直接写 `await`（如 `await agent.ainvoke(...)`），这在 Jupyter（IPython 7+）里可以直接运行；如果你把代码复制到 `.py` 文件单独跑，需要把它包进 `async def main(): ...` 再用 `asyncio.run(main())` 调用，否则会报 `SyntaxError`。

&emsp;&emsp;铺垫到这里就够了，我们直接进入第二部分的第一步：进化的一切，都建立在「Agent 能记住什么」之上。

## <center>第5章：三层记忆——SOUL / MEMORY / SqliteSaver</center>

&emsp;&emsp;开篇我们说过，进化的地基是「Agent 能分层记住什么」。这一章我们就把这个基础打好。我们会先理解 `FuFan-OpenHermes` 为什么把记忆分成三层、每一层各管什么，然后用大约四十行 Python 复现它的核心读写与备份逻辑，并跑一遍亲眼看到备份机制生效——`.bak` 里稳稳存着被覆盖的上一版。

&emsp;&emsp;这一章没有大模型调用，全部是纯文件操作，所以它也是整门课里最适合用来熟悉「源码锚点对照」这种学习方式的一章。我们写的每一段代码，旁边都会贴上项目源码里的对应行号，让你亲眼看到「我写的」和「项目里的」是同一个东西。这一章产出的 `MiniMemoryOps` 实例和它的备份机制，会在第 11 章被真正用上，所以请你把它学扎实。

### 5.1 为什么是三层，而不是一个向量库

&emsp;&emsp;提到「给 Agent 加记忆」，常见做法是接入一个向量数据库。向量库当然强大，但 `FuFan-OpenHermes` 在记忆这件事上做了一个克制的选择：**用纯文件系统，分成三层存储**。这里我们把它称为「三层记忆」——**需要强调的是，「三层记忆」是我们这门课对项目记忆结构的提炼命名，项目源码里并没有一个叫「三层记忆」的统一术语**，但 `SOUL.md`、`MEMORY.md`、`SqliteSaver` 这三个物理存在的载体，确实构成了三个清晰的层次。

&emsp;&emsp;我们先看清楚这三层各自的职责，再讨论为什么不用向量库。

<p align="center"><font face="黑体" size=4>三层记忆的职责分工</font></p>
<div class="center">
<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>

| 层级 | 物理载体 | 职责 | 变更频率 |
|------|---------|------|---------|
| 人格层 | `SOUL.md` | 定义 Agent 是谁、倾向怎么回答 | 极低，几乎不变 |
| 长期记忆层 | `MEMORY.md` | 沉淀跨会话的用户偏好、事实 | 低，定期整理 |
| 会话层 | `SqliteSaver`（`state.db`）| 保存单次会话的消息历史与状态 | 高，每轮都写 |

</div>

&emsp;&emsp;为什么是文件系统而不是向量库？因为这三层记忆的核心诉求是**可读、可编辑、可审计**。`SOUL.md` 和 `MEMORY.md` 是纯 Markdown 文本，你随时可以打开看一眼 Agent「记住了什么」，甚至手工改一行——而这正是第 11 章「AI 优化记忆」能够安全落地的前提。向量库把记忆变成了一堆不可读的浮点数，你没法直接看、没法直接改、也没法直接信任。在「让 Agent 管理自己的记忆」这个目标下，文件系统的透明性远比向量检索的语义能力更重要。

&emsp;&emsp;这里要澄清一个容易混淆的点：**「不用向量库」不等于「不做检索」**。随着会话越来越多，Agent 迟早需要「跨会话回忆」——记起自己在别的会话里和用户聊过什么。这个检索需求真实存在，但项目解决它用的依然不是向量库，而是 `SQLite` 自带的 `FTS5` 全文检索——它一样可读、可审计、零额外运维。换句话说，检索 ≠ 必须上向量库。这一层我们留到本章最后的 5.5 节专门复现，先把前三层的「存」打扎实。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165948869.png" width=50%></div>

### 5.2 复现 MiniMemoryOps：读写与备份的最小实现

&emsp;&emsp;理解了分层设计，我们就来动手复现它的核心代码。`FuFan-OpenHermes` 把记忆的读写操作集中在一个 `MemoryOps` 类里（源码 `agent/memory_ops.py:16`），它负责 `SOUL.md` 和 `MEMORY.md` 的读、写、备份和 token 估算。我们先建一个临时的会话目录，再把这个类用四十行左右复现出来。

&emsp;&emsp;接下来这段代码做的是搭一个完全自包含的实验环境——我们用 `tempfile` 建一个临时 session 目录，不依赖项目工程，跑完即可清理。**运行后你会看到打印出的临时目录路径**，后面所有记忆操作都在这个目录里发生。

In [1]:
# 搭建自包含的实验环境：用临时目录模拟一个 Agent 会话目录
import tempfile
from pathlib import Path

# 建一个临时 session 目录（跑完即弃，不污染你的工程）
SESSION_DIR = Path(tempfile.mkdtemp(prefix="hermes_session_"))
print("本次实验的会话目录：", SESSION_DIR)

本次实验的会话目录： /var/folders/fl/8wq5_lz53ln9ypplts4z_1tr0000gn/T/hermes_session_9f7ewtf1


&emsp;&emsp;有了实验目录，我们就来写 `MiniMemoryOps`。它对照的是项目源码 `agent/memory_ops.py` 的 `MemoryOps` 类：`approx_tokens` 对应源码第 7 行的 token 估算、`read_soul/write_soul` 对应第 22-26 行、`_backup_then_write` 对应第 40-43 行的备份逻辑。我们把这些方法原样搬过来，只去掉了项目里的 `safe_read_text` 编码兜底（用标准 `read_text` 替代），核心算法一字不改。

In [2]:
# 复现项目 agent/memory_ops.py 的 MemoryOps —— 三层记忆中"可读可编辑"两层的读写引擎
class MiniMemoryOps:
    """会话级记忆操作：SOUL.md / MEMORY.md 的读写 + 写前备份。

    对照源码 agent/memory_ops.py:16（MemoryOps 类）。
    """

    def __init__(self, session_dir: Path) -> None:
        # 两个记忆文件的路径（对照源码 19-20 行）
        self.session_dir = Path(session_dir)
        self.soul = self.session_dir / "SOUL.md"      # 人格层
        self.memory = self.session_dir / "MEMORY.md"  # 长期记忆层

    def approx_tokens(self, s: str) -> int:
        """快速估算 token 数（对照源码第 7 行）。

        规则：中文(CJK)约 1.5 字符 1 token，其它约 4 字符 1 token。
        """
        if not s:
            return 0
        # 统计 CJK 字符数（中日韩统一表意文字区间）
        cjk = sum(1 for c in s if "一" <= c <= "鿿")
        other = len(s) - cjk
        return int(cjk / 1.5 + other / 4)

    def read_soul(self) -> str:
        # 文件不存在返回空串，不抛异常（对照源码 22-23 行）
        return self.soul.read_text(encoding="utf-8") if self.soul.exists() else ""

    def write_soul(self, content: str) -> None:
        # 写之前先备份（对照源码 25-26 行，委托 _backup_then_write）
        self._backup_then_write(self.soul, content)

    def read_memory(self) -> str:
        return self.memory.read_text(encoding="utf-8") if self.memory.exists() else ""

    def write_memory(self, content: str) -> None:
        self._backup_then_write(self.memory, content)

    def stats(self) -> dict:
        # 返回两层记忆的 token 估算，便于上下文预算监控（对照源码 34-38 行）
        return {
            "soul_tokens": self.approx_tokens(self.read_soul()),
            "memory_tokens": self.approx_tokens(self.read_memory()),
        }

    def _backup_then_write(self, p: Path, content: str) -> None:
        """备份机制核心（对照源码 40-43 行）：若目标文件已存在，先复制为 .bak，再写入新内容。"""
        if p.exists():
            # 旧内容备份到 同名 + .bak，这是第 11 章 AI 优化记忆能"反悔"的前提
            (p.parent / f"{p.name}.bak").write_text(
                p.read_text(encoding="utf-8"), encoding="utf-8"
            )
        p.write_text(content, encoding="utf-8")

&emsp;&emsp;这段代码最值得你停下来看的是 `_backup_then_write`。它的逻辑很简单：写新内容之前，如果文件已经存在，就先把旧内容复制一份成 `.bak`。看起来不起眼，但这正是整套记忆系统敢于「让大模型破坏性重写 `MEMORY.md`」的安全底线——因为只要有 `.bak`，万一 AI 把记忆改坏了，你随时能回滚。这个设计在第 11 章 AI 优化记忆时会真正用到。

### 5.3 跑一遍：备份机制是怎么生效的

&emsp;&emsp;光看代码不够，我们要亲眼看到备份机制生效。下面我们做一个对比实验：第一次写 `SOUL.md`，此时还没有 `.bak`；第二次再写，旧内容就应该被备份到 `.bak` 里。**运行后你会看到两次写入后目录内容的变化——第二次写入后 `SOUL.md.bak` 出现了，且它的内容正是第一次写入的值**。

In [3]:
# 对比实验：第一次写 vs 第二次写，观察 .bak 备份的产生
ops = MiniMemoryOps(SESSION_DIR)

# 第一次写入人格设定（此前文件不存在，不会产生 .bak）
ops.write_soul("我是一个研究助手，倾向于简洁、直接地回答问题。")
print("第一次写入后，目录里的文件：", [p.name for p in SESSION_DIR.iterdir()])

# 第二次写入（覆盖人格设定，此时旧内容应被备份为 SOUL.md.bak）
ops.write_soul("我是一个研究助手，回答时会先给结论，再补充推理过程。")
print("第二次写入后，目录里的文件：", [p.name for p in SESSION_DIR.iterdir()])

# 验证 .bak 里存的是不是"第一次写入的旧内容"
bak = (SESSION_DIR / "SOUL.md.bak").read_text(encoding="utf-8")
print("\n.bak 备份的内容（应为第一次写入的值）：\n", bak)
print("\n当前 SOUL.md 的内容（应为第二次写入的值）：\n", ops.read_soul())

第一次写入后，目录里的文件： ['SOUL.md']
第二次写入后，目录里的文件： ['SOUL.md.bak', 'SOUL.md']

.bak 备份的内容（应为第一次写入的值）：
 我是一个研究助手，倾向于简洁、直接地回答问题。

当前 SOUL.md 的内容（应为第二次写入的值）：
 我是一个研究助手，回答时会先给结论，再补充推理过程。


&emsp;&emsp;运行结果会让你看得很清楚：第一次写入后目录里只有 `SOUL.md`；第二次写入后多出了一个 `SOUL.md.bak`，而且 `.bak` 里存的恰好是被覆盖掉的第一版内容。这就是「写前备份」语义——任何一次覆盖写，都会把上一版留存下来。接下来我们再看 `approx_tokens` 的估算行为，它体现了中英文的差异。

&emsp;&emsp;下面这段验证 token 估算对中文和英文的不同处理。**运行后你会看到同样字符数的中文和英文，估算出的 token 数不同**——因为源码假设中文信息密度更高（1.5 字符 1 token），英文更低（4 字符 1 token）。

In [4]:
# 验证 approx_tokens：中文 vs 英文的估算差异
chinese = "今天天气很好我们一起去公园散步吧"   # 16 个中文字
english = "the weather is really nice today go"  # 35 个英文字符
print(f"中文 {len(chinese)} 字 → 估算 {ops.approx_tokens(chinese)} tokens")
print(f"英文 {len(english)} 字 → 估算 {ops.approx_tokens(english)} tokens")
print("当前记忆统计：", ops.stats())

中文 16 字 → 估算 10 tokens
英文 35 字 → 估算 8 tokens
当前记忆统计： {'soul_tokens': 16, 'memory_tokens': 0}


&emsp;&emsp;这里能看到中文的 token 估算明显高于同字符数的英文，这符合「中文每个字承载的信息更密」的经验事实。`stats()` 则把两层记忆的 token 数打包返回，在真实项目里它用于监控上下文预算——当记忆膨胀到一定 token 数，就该触发第 11 章的「整理记忆」了。

### 5.4 会话层 SqliteSaver 的设计思想

&emsp;&emsp;三层记忆里，前两层（`SOUL.md` / `MEMORY.md`）我们已经用代码复现了，第三层「会话层」我们只需要理解它的设计思想，不必完整实现——因为它本质上是 `LangGraph` 提供的标准能力，项目源码 `agent/agent_manager.py:83` 用 `AsyncSqliteSaver.from_conn_string(...)` 把每个会话的消息历史持久化到一个独立的 `state.db` 里。

&emsp;&emsp;它的关键设计有两点。第一，**每个会话独立一个 sqlite 文件**，会话之间互不干扰，这让多会话并发成为可能。第二，**它和前两层共享「纯文件系统」的哲学**——sqlite 文件躺在磁盘上，你可以用任何 sqlite 工具打开它审计，不像向量库那样是个黑盒。这三层合起来，就构成了一个完全透明、可审计、可手工干预的记忆系统，为后面的自我进化提供了坚实又可信的地基。

&emsp;&emsp;三层记忆到这里就复现完了——前两层用 `MiniMemoryOps` 亲手实现，5.3 里你已经亲眼看到备份机制生效（`.bak` 稳稳存着被覆盖的上一版、当前文件是新版），第三层理解了 `SqliteSaver` 的透明设计。但这三层有一个共同的边界：**它们都是「单会话内」的记忆**——`SOUL.md`、`MEMORY.md`、`state.db` 都躺在 `sessions/{sid}/` 这个会话专属目录里。一旦换了个会话，Agent 就「失忆」了，记不起自己在别的会话里和用户聊过什么。要让 Agent 跨会话回忆，就需要 5.1 节预告过的那一层——跨会话检索。这是我们打地基的最后一块拼图。

### 5.5 跨会话记忆检索：FTS5 双索引表

&emsp;&emsp;前面三层解决的是「单次会话内记住什么」，这一节解决的是「跨会话回忆」：Agent 在第 100 个会话里，怎么记起它在第 3 个会话里学到的某个事实？项目的答案不是向量库，而是 `SQLite` 自带的 `FTS5`（Full-Text Search 第 5 版）全文检索。它对照的是项目源码 `agent/memory_index.py`（277 行）——一个**全局、跨会话** 的记忆索引，物理上是 `storage/memory.db` 这个单一的 sqlite 文件（注意：是 `.db` 数据库，不是某个 `.md` 文本；它由 `app.py:24` 的 `MemoryIndex(STORAGE_DIR / "memory.db")` 在启动时创建，全项目共用一个实例）。

&emsp;&emsp;这一节和前面几节同性质——**纯 `sqlite` 操作、零大模型调用**，复制进 Notebook 就能跑。它在项目里怎么真正接进 Agent，简单说是两侧：写入侧由对话流程把每条消息 `index_message` 进库，检索侧则被包装成一个 Agent 可主动调用的跨会话检索工具（`session_search`，见项目 `tools/__init__.py`）。这部分集成涉及 Agent 内核与工具装配，超出本节范围；本节先把这个检索引擎本身复现扎实。

#### 5.5.1 为什么是「双表」：英文 BM25 + 中文 trigram

&emsp;&emsp;全文检索最直接的做法是建一张 `FTS5` 表，这对英文确实够用，但中文会出问题。`FTS5` 默认的 `unicode61` 分词器是按空格和标点切词的，英文「python programming」会被切成 `python`、`programming` 两个词，检索很自然；可中文「大语言模型」中间没有空格，`unicode61` 会把一整句当成一个无法再切的 token，你搜「语言模型」根本命中不了「大语言模型」这条记录。

&emsp;&emsp;项目的解法是**建两张 `FTS5` 表**（源码 `memory_index.py:28,43`），同一份内容索引两遍，检索时按语言走不同的表。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>FTS5 双索引表的分工</font></p>
<div class="center">

| 索引表 | 分词器 | 适合的语言 | 检索方式 |
|--------|--------|-----------|---------|
| `messages_fts` | `unicode61`（默认）| 英文等带空格语言 | BM25 相关性排序 |
| `messages_fts_trigram` | `trigram`（三字滑窗）| 中文等无空格 CJK | 子串匹配 |

</div>

&emsp;&emsp;`trigram` 的原理很巧：它把文本按「每 3 个字符」滑动切片，「大语言模型」会被切成 `大语言`、`语言模`、`言模型` 这些三字片段建索引——这样你搜「语言模型」时，它的三字片段能和索引对上，子串检索就成立了。代价是 `trigram` 要求查询词至少 3 个字符，更短的查询它覆盖不了（这个边界我们 5.5.3 会再说）。

> **【踩坑预警】**：`trigram` 分词器需要 `SQLite ≥ 3.34`（2020 年底发布）。绝大多数现代 Python 自带的 `sqlite3` 都满足（本课实测环境是 `3.51.2`），但如果你在很老的系统上跑，建虚拟表时会报 `no such tokenizer: trigram`。排查方法：`python -c "import sqlite3; print(sqlite3.sqlite_version)"` 看版本；低于 3.34 就升级 Python 或系统的 `sqlite` 库。

#### 5.5.2 复现 MiniMemoryIndex：建表 + trigger 自动同步

&emsp;&emsp;理解了双表，我们就来建库。这里有一个 `FTS5` 的经典用法值得你掌握——**用 `trigger`（触发器）让索引表自动跟着主表同步**。我们把消息原文存在一张普通的 `messages` 主表里，再为它配两张 `FTS5` 虚拟表；然后建 `trigger`，每当 `messages` 插入一行，数据库就自动把内容同步进两张索引表。这样业务代码只管往主表写，索引「自动」就建好了，不用手动维护两张表的一致性。

&emsp;&emsp;下面这段对照源码 `memory_index.py:19-93`，把建表 SQL（含 `trigger`）和写入方法 `index_message` 复现出来。为聚焦教学，我们简化掉了源码里的线程锁（`threading.Lock`）和 `update/delete` 两组 `trigger`，只保留最能说明问题的 `insert` 同步——核心的「主表 + 双 FTS5 表 + trigger 自动同步」结构和源码完全一致。

In [5]:
# 复现项目 agent/memory_index.py 的 MemoryIndex —— 跨会话 FTS5 双索引（对照源码 19-93 行）
import sqlite3
import time
from pathlib import Path

# 建表 SQL：1 张主表 + 2 张 FTS5 虚拟表 + 各自的 insert 同步 trigger（对照源码 19-57 行）
_CREATE_SQL = """
CREATE TABLE IF NOT EXISTS messages (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT NOT NULL,
    role TEXT NOT NULL,
    content TEXT NOT NULL,
    ts REAL NOT NULL
);
-- 英文表：默认 unicode61 分词器，走 BM25 相关性排序
CREATE VIRTUAL TABLE IF NOT EXISTS messages_fts USING fts5(content);
CREATE TRIGGER IF NOT EXISTS messages_fts_insert AFTER INSERT ON messages BEGIN
    INSERT INTO messages_fts(rowid, content) VALUES (new.id, COALESCE(new.content, ''));
END;
-- 中文表：trigram 三字滑窗分词器，支持 CJK 子串检索
CREATE VIRTUAL TABLE IF NOT EXISTS messages_fts_trigram USING fts5(content, tokenize='trigram');
CREATE TRIGGER IF NOT EXISTS messages_fts_trigram_insert AFTER INSERT ON messages BEGIN
    INSERT INTO messages_fts_trigram(rowid, content) VALUES (new.id, COALESCE(new.content, ''));
END;
"""


class MiniMemoryIndex:
    """全局跨会话 FTS5 记忆索引（对照源码 memory_index.py:60 MemoryIndex 类）。"""

    def __init__(self, db_path) -> None:
        self._conn = sqlite3.connect(str(db_path), check_same_thread=False)
        self._conn.row_factory = sqlite3.Row  # 让查询结果能按列名取值
        self._conn.executescript(_CREATE_SQL)  # 一次性建好主表 + 双索引 + trigger
        self._conn.commit()

    def index_message(self, session_id: str, role: str, content: str) -> None:
        """写一条消息进主表；trigger 会自动同步到两张 FTS5 表（对照源码 84-93 行）。"""
        if not content or not content.strip():
            return  # 空内容不索引
        # 只往主表写，两张索引表由 trigger 自动跟进——这是 FTS5 + trigger 的核心用法
        self._conn.execute(
            "INSERT INTO messages(session_id, role, content, ts) VALUES (?,?,?,?)",
            (session_id, role, content, time.time()),
        )
        self._conn.commit()

    @staticmethod
    def _contains_cjk(text: str) -> bool:
        """判断文本是否含中日韩字符，用于检索时选表（对照源码 _contains_cjk:133）。"""
        return any("一" <= ch <= "鿿" for ch in text)

print("MiniMemoryIndex 定义就绪：1 主表 + 2 FTS5 索引表 + trigger 自动同步")

MiniMemoryIndex 定义就绪：1 主表 + 2 FTS5 索引表 + trigger 自动同步


&emsp;&emsp;这段代码的精华是那两个 `trigger`。`index_message` 只往 `messages` 主表插一行，但 `AFTER INSERT` 触发器会立刻把同一份 `content` 写进 `messages_fts` 和 `messages_fts_trigram` 两张索引表。这意味着「写入」和「建索引」被解耦了——你的业务代码完全不用关心索引的存在，只管写主表，检索能力自动就有了。`_contains_cjk` 这个小工具是给下一步检索用的：它决定一个查询该走英文表还是中文表。

#### 5.5.3 复现 search：中英文双路检索

&emsp;&emsp;库建好了，来写检索。`search` 的核心逻辑是**按查询语言选表**（对照源码 `memory_index.py:154-277`）：查询里含中文就走 `trigram` 表做子串匹配，纯英文就走默认表走 BM25 排序。两条路都用 `FTS5` 的 `MATCH` 操作符查询，并用 `snippet()` 函数把命中的关键词用 `>>>` `<<<` 高亮出来，方便后面喂给大模型时它能一眼看到命中点。

&emsp;&emsp;下面把 `search` 方法补全。需要诚实说明的是：源码的 `search` 是**三路**——除了中文 `trigram` 和英文 BM25，还有一条针对「短中文（少于 3 字）」的 `LIKE` 兜底路径（因为 `trigram` 覆盖不了少于 3 字的查询），以及一段对用户输入做转义的 `_sanitize_fts5_query`。我们这里**简化为两路**（中文 trigram + 英文 BM25），把短中文兜底和输入转义作为踩坑点提示，不实现——核心的「按语言选表 + MATCH + snippet 高亮」和源码一致。

In [6]:
# 给 MiniMemoryIndex 补上 search —— 按语言选表的双路检索（对照源码 memory_index.py:154-277）
from typing import Optional

def search(self, query: str, top_k: int = 5, exclude_session: Optional[str] = None) -> list[dict]:
    """跨会话检索：含中文走 trigram 表，纯英文走 BM25 表。

    返回 [{session_id, role, content, snippet, rank}, ...]。
    """
    if not query or not query.strip():
        return []  # 空查询直接返回空

    if self._contains_cjk(query):
        # 中文路径：trigram 表，查询整体加双引号作为短语匹配
        table = "messages_fts_trigram"
        match = '"' + query.strip().replace('"', '""') + '"'
    else:
        # 英文路径：默认 unicode61 表，FTS5 自动按词 + BM25 排序
        table = "messages_fts"
        match = query.strip()

    where = [f"{table} MATCH ?"]
    params: list = [match]
    if exclude_session:
        # 跨会话召回时，通常要排除「当前会话自己」，只回忆别的会话（对照源码 exclude_session 参数）
        where.append("m.session_id != ?")
        params.append(exclude_session)
    params.append(top_k)

    # snippet(表, 列, 左标记, 右标记, 省略号, 最多token数)：把命中处高亮，便于喂给大模型
    sql = f"""
        SELECT m.session_id, m.role, m.content,
               snippet({table}, 0, '>>>', '<<<', '...', 15) AS snippet,
               rank
        FROM {table} JOIN messages m ON m.id = {table}.rowid
        WHERE {' AND '.join(where)}
        ORDER BY rank
        LIMIT ?
    """
    cur = self._conn.execute(sql, params)
    return [dict(r) for r in cur.fetchall()]

# 把 search 方法挂到类上（教学写法，等价于写在 class 体内）
MiniMemoryIndex.search = search
print("search 方法就绪：中文走 trigram 子串匹配，英文走 BM25 相关性排序")

search 方法就绪：中文走 trigram 子串匹配，英文走 BM25 相关性排序


&emsp;&emsp;这段 `search` 最值得看的是 `exclude_session` 参数：跨会话召回时，我们通常想「回忆**别的** 会话里说过什么」，而不是把当前会话自己的话又翻出来，所以可以传入当前 `session_id` 把它排除掉——这正是跨会话召回时要用的（用当前会话的 user 消息去查别的会话）。`snippet()` 则把命中的关键词高亮成 `>>>关键词<<<`，让检索结果对大模型更友好。

> **【踩坑预警】**：`trigram` 查询要求至少 3 个字符，所以「记忆」「部署」这种 2 字中文查询，走 `trigram` 表会命中不全甚至查不到。源码为此专门加了一条 `LIKE` 兜底路径（`memory_index.py:218-252`）：短中文降级用 `content LIKE '%关键词%'` 直接子串扫描。我们的最小复现版简化掉了这条路径，所以**演示时请用 3 字及以上的中文查询**（如「大语言模型」）；真实项目里短查询由 `LIKE` 兜底，不会漏。另外源码还有 `_sanitize_fts5_query`（`:99-120`）对用户输入里的 `FTS5` 保留符号（`+ * " ^` 等）做转义，防止用户输入触发语法错误——生产环境这步不能省。

#### 5.5.4 跑一遍：写入跨会话消息，中英文双路检索

&emsp;&emsp;光看不够，我们建一个库，写入几条来自不同会话的中英文消息，再分别用中文和英文查询，亲眼看检索效果。**运行后你会看到：中文查「大语言模型」命中对应记录、英文查「python」命中对应记录，且命中处都被 `>>><<<` 高亮**。

In [7]:
# 跑一遍：写入多会话消息 + 中英文双路检索
import tempfile

db_path = Path(tempfile.mkdtemp(prefix="hermes_memidx_")) / "memory.db"
idx = MiniMemoryIndex(db_path)
print("记忆索引库已建在：", db_path)

# 写入来自 3 个不同会话的消息（模拟跨会话历史）
idx.index_message("sess_01", "user", "今天我们讨论大语言模型的应用场景，包括智能助手和代码生成")
idx.index_message("sess_02", "assistant", "Python programming is very useful for data science")
idx.index_message("sess_03", "user", "我在用大语言模型做一个客服机器人")

print("\n[中文检索] 查「大语言模型」：")
for h in idx.search("大语言模型"):
    print(f"  [{h['role']}@{h['session_id']}] {h['snippet']}")

print("\n[英文检索] 查「python」：")
for h in idx.search("python"):
    print(f"  [{h['role']}@{h['session_id']}] {h['snippet']}")

记忆索引库已建在： /var/folders/fl/8wq5_lz53ln9ypplts4z_1tr0000gn/T/hermes_memidx_2v5mzlrg/memory.db

[中文检索] 查「大语言模型」：
  [user@sess_03] 我在用>>>大语言模型<<<做一个客服机器人
  [user@sess_01] 今天我们讨论>>>大语言模型<<<的应用场景，...

[英文检索] 查「python」：
  [assistant@sess_02] >>>Python<<< programming is very useful for data science


&emsp;&emsp;你会看到中文查询命中了 `sess_01` 和 `sess_03` 两条含「大语言模型」的记录（来自不同会话——这正是「跨会话」的意义），英文查询命中了 `sess_02` 的 Python 记录，命中关键词都被高亮。同一份内容、不同语言的查询各走各的表，互不干扰。下面用两层验证把这个检索引擎钉死。

&emsp;&emsp;Tier 1 验证「形」——双表是否都建出来了、写一条主表后两张索引表是否都被 `trigger` 自动同步了。**运行后你会看到三张表都存在，且写入 1 条后两张 FTS5 表各自都有 1 行**。

In [8]:
# Tier 1（组件级）：验证双表建立 + trigger 自动同步
_db = Path(tempfile.mkdtemp()) / "t1.db"
_idx = MiniMemoryIndex(_db)

# 检查三张表都存在
_tables = {r[0] for r in _idx._conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'").fetchall()}
print("messages 主表存在：       ", "messages" in _tables)
print("messages_fts 英文表存在：  ", "messages_fts" in _tables)
print("messages_fts_trigram 表存在：", "messages_fts_trigram" in _tables)

# 写 1 条，验证两张 FTS5 表都被 trigger 自动同步
_idx.index_message("s", "user", "深度学习模型训练")
_n_main = _idx._conn.execute("SELECT count(*) FROM messages").fetchone()[0]
_n_fts = _idx._conn.execute("SELECT count(*) FROM messages_fts").fetchone()[0]
_n_tri = _idx._conn.execute("SELECT count(*) FROM messages_fts_trigram").fetchone()[0]
print(f"\n写入 1 条后：主表 {_n_main} 行，英文索引 {_n_fts} 行，trigram 索引 {_n_tri} 行")

# 机器门控哨兵
assert {"messages", "messages_fts", "messages_fts_trigram"} <= _tables
assert _n_main == _n_fts == _n_tri == 1
print("\n[PASS] 双表建立 + trigger 自动同步生效")

messages 主表存在：        True
messages_fts 英文表存在：   True
messages_fts_trigram 表存在： True

写入 1 条后：主表 1 行，英文索引 1 行，trigram 索引 1 行

[PASS] 双表建立 + trigger 自动同步生效


&emsp;&emsp;Tier 1 确认了索引结构正确。Tier 2 验证「神」——检索真的能跨会话命中，且 `exclude_session` 能把指定会话排除掉。这是检索引擎最关键的两条不变量。**运行后你会看到中英文检索各命中 ≥1 条，且排除某会话后结果里不再有它**。

In [9]:
# Tier 2（端到端）：跨会话命中 + exclude_session 排除
_db2 = Path(tempfile.mkdtemp()) / "t2.db"
_idx2 = MiniMemoryIndex(_db2)
_idx2.index_message("A", "user", "我们正在研究大语言模型的推理能力")
_idx2.index_message("B", "user", "大语言模型在多个会话里都被提到")
_idx2.index_message("C", "assistant", "machine learning needs good data")

# 中文跨会话命中
_cn = _idx2.search("大语言模型")
print("中文查「大语言模型」命中数：", len(_cn), "（预期 ≥2，来自 A、B 两个会话）")
# 英文命中
_en = _idx2.search("machine learning")
print("英文查「machine learning」命中数：", len(_en), "（预期 ≥1）")
# 排除会话 A：结果里不应再有 A
_excl = _idx2.search("大语言模型", exclude_session="A")
_sids = {h["session_id"] for h in _excl}
print("排除 A 后命中的会话：", _sids, "（预期不含 A）")

# 机器门控哨兵
assert len(_cn) >= 2 and len(_en) >= 1 and "A" not in _sids
print("\n[PASS] 跨会话检索命中正确，exclude_session 排除生效")

中文查「大语言模型」命中数： 2 （预期 ≥2，来自 A、B 两个会话）
英文查「machine learning」命中数： 1 （预期 ≥1）
排除 A 后命中的会话： {'B'} （预期不含 A）

[PASS] 跨会话检索命中正确，exclude_session 排除生效


&emsp;&emsp;两层验证通过，意味着我们手里这个 `MiniMemoryIndex` 和项目源码的 `MemoryIndex` 是同一套机制：双表索引、`trigger` 自动同步、按语言双路检索、可按会话排除。地基的最后一块拼图到位了——Agent 现在不仅能在单会话里分层记忆，还能跨会话检索回忆。

&emsp;&emsp;但到这里，所有这些记忆（三层文件 + FTS5 索引）都还只是「存」和「能查」，大模型并不会自动用上它们。下一章我们先解决第一个衔接：单会话的三层记忆，是怎么被分层拼成一段系统提示词喂给模型的。（本节的 FTS5 检索结果在项目里有专门的去处——`build_system_prompt` 里的 `Recalled` 注入层，以及上面提到的 `session_search` 工具；这两条接入路径都属于 Agent 内核与工具装配层，本课主线点到为止，下一章聚焦三层记忆的分层拼接本身。）

## <center>第6章：System Prompt 分层拼接——三层记忆怎么喂给模型</center>

&emsp;&emsp;上一章我们把三层记忆的「存」实现了——`SOUL.md`、`MEMORY.md`、`SqliteSaver` 各司其职。但这里有一个容易被忽略的断层：这些记忆此刻只是静静躺在磁盘上的 Markdown 文件，大模型并不会自动去读它们。**谁来在每一轮对话开始时，把这些文件「装配」成大模型真正看得到的那段系统提示词？** 这就是本章的主角——`build_system_prompt`。

&emsp;&emsp;它做的事一句话能说清：每轮对话开始时，按固定顺序把会话目录里的几个文件读出来、分层拼成一整段 `system_prompt`，再交给大模型。这一章和上一章同性质——纯文件读取、零大模型调用，复制进 Notebook 就能跑。但它有两个特别值得看清的设计：一是**分层拼接的固定顺序**（人格在前、协议在后，不是随便排的），二是**技能的两级加载**（清单 vs 全文，一种「渐进式披露」）。还有一个贯穿始终的特性——**热更新**：因为每轮都重新读文件，你改了 `MEMORY.md`，下一轮提示词立刻就变，不需要重启服务。

### 6.1 源码锚点对照

&emsp;&emsp;先看真实源码的全貌。`agent/prompt_builder.py` 全文只有 63 行，结构非常清爽，关键区段如下表。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>prompt_builder.py 关键区段对照</font></p>
<div class="center">

| 源码区段 | 行号 | 作用 |
|---------|------|------|
| `MAX_LAYER_CHARS` | 第 7 行 | 单层超长截断阈值（20000 字符）|
| `PROTOCOL_FOOTER` | 第 9-19 行 | 固定追加在末尾的协议说明（工作区 / 记忆 / 技能的使用约定）|
| `_read` | 第 22-28 行 | 读单层文件：不存在返回空串，超长则截断 |
| `build_system_prompt` | 第 31-63 行 | 主体：按层读文件、拼接、收尾 |

</div>

&emsp;&emsp;主体的拼接顺序是写死的：`SOUL`（第 36-38 行）→ `MEMORY`（第 40-42 行）→ `SKILLS_SNAPSHOT`（第 44-46 行）→ 已加载技能全文（第 49-60 行）→ `PROTOCOL_FOOTER`（第 62 行）。这个顺序不是随意的：**人格（我是谁）放最前定调，长期记忆（我记得什么）紧随其后，技能清单（我会什么）再往后，已激活技能的完整说明书放在技能清单之后，协议（怎么用工具）压轴收尾**——从「身份」到「记忆」到「能力」到「行动规约」，是一条从稳定到具体的递进。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165952303.png" width=50%></div>

### 6.2 分层拼接器实现

&emsp;&emsp;理解了顺序，我们就来复现它。下面这段代码对照源码 `build_system_prompt`（第 31-73 行），把分层装配逻辑搬过来。我们做了三处便于教学的最小改写：把项目里的 `safe_read_text` 编码兜底换成标准 `read_text`；给函数名加后缀（`_read`→`_read_layer`、`build_system_prompt`→`build_system_prompt_mvp`）以示这是教学复现版；以及把 `PROTOCOL_FOOTER` 定义去掉源码里的前导换行，于是源码第 72 行那次 `.strip()` 在我们这里就不再需要了。还有一点要如实说明：项目源码在「长期记忆」层和「技能快照」层之间其实还有两层——`Global Memory`（跨项目共享的全局记忆）和 `Recalled`（5.3 节 FTS5 跨会话检索结果的注入位）；为聚焦「单会话三层记忆怎么喂给模型」这条主线，本课复现版从略这两层，只保留最能说明分层拼接思想的五层，其余逻辑完全对照源码。注意两个细节：每层前面都贴一个 `` 的 HTML 注释标签（让大模型能清楚区分「这段是人格」还是「这段是记忆」），以及单层超过 20000 字符就截断（防止某个文件失控撑爆上下文）。

In [16]:
# 复现项目 agent/prompt_builder.py 的 build_system_prompt —— 把三层记忆 + 技能装配成 system_prompt
import json
from pathlib import Path

MAX_LAYER_CHARS = 20_000  # 单层超长截断阈值（对照源码第 7 行）

# 协议 footer：固定追加在末尾，告诉 Agent 工作区/记忆/技能的使用约定（对照源码第 9-19 行，保留英文原文）
PROTOCOL_FOOTER = """## Protocol

- The user's working directory is the **workspace**. `terminal`, `python_repl`,
  `read_file`, `write_file` operate inside it.
- `MEMORY.md` is your long-term notebook. Use `write_memory` for facts that
  should survive across turns.
- `SKILLS_SNAPSHOT.md` lists available skills. If one looks relevant, call
  `read_skill('<name>')` to load its full content.
- Be concise. Cite tool results, don't paraphrase them."""


def _read_layer(p: Path) -> str:
    """读单层文件：不存在返回空串，超长则截断（对照源码 _read 第 22-28 行）。"""
    if not p.exists():
        return ""
    c = p.read_text(encoding="utf-8")
    if len(c) > MAX_LAYER_CHARS:
        c = c[:MAX_LAYER_CHARS] + "\n...[truncated]"
    return c


def build_system_prompt_mvp(session_dir: Path) -> str:
    """把会话目录里的记忆/技能文件分层拼成一段 system_prompt（对照源码 build_system_prompt 第 31-73 行）。

    顺序：SOUL → MEMORY → SKILLS_SNAPSHOT(Level 0) → 已加载技能全文(Level 1) → Protocol footer。
    """
    sd = Path(session_dir)
    parts: list[str] = []

    # 第 1 层：人格（对照源码 36-38 行）
    soul = _read_layer(sd / "SOUL.md")
    if soul:
        parts.append(f"\n{soul}")

    # 第 2 层：长期记忆（对照源码 40-42 行）
    memory = _read_layer(sd / "MEMORY.md")
    if memory:
        parts.append(f"\n{memory}")

    # 第 3 层：技能快照 Level 0——只列清单，不含技能正文（对照源码 54-56 行）
    snap = _read_layer(sd / "skills" / "SKILLS_SNAPSHOT.md")
    if snap:
        parts.append(f"\n{snap}")

    # 第 4 层：已加载技能 Level 1——注入完整 SKILL.md 正文（对照源码 59-70 行）
    loaded_path = sd / "loaded_skills.json"
    if loaded_path.exists():
        try:
            loaded = json.loads(loaded_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            loaded = []  # 解析失败静默兜底：技能全文不注入，但不报错（对照源码 62-64 行）
        for item in loaded:
            if item.get("level") != 1:
                continue  # 只注入 level=1 的技能，清单里其它技能仍停留在 Level 0
            full = _read_layer(sd / "skills" / item["name"] / "SKILL.md")
            if full:
                parts.append(f"\n{full}")

    # 第 5 层：协议 footer，固定收尾（对照源码 72 行）
    parts.append(PROTOCOL_FOOTER)

    # 各层之间用双换行分隔（对照源码 73 行 return）
    return "\n\n".join(parts)

&emsp;&emsp;这段代码里最值得停下来看的是第 4 层——**技能的两级加载**。`SKILLS_SNAPSHOT.md`（Level 0）只列一份「技能清单」：每个技能的名字 + 一句话描述，让大模型知道「我有哪些技能可用」；而只有当某个技能被标记为已加载（`loaded_skills.json` 里 `level=1`），它的完整 `SKILL.md` 正文才会被注入提示词。这是一种刻意的「渐进式披露」：不把所有技能的说明书一股脑塞进上下文（那会非常占 token），而是先给清单、用到哪个再拉哪个的全文。需要说明的是——`SKILLS_SNAPSHOT.md` 和 `skills/` 目录里的技能文件是从哪来的？它们是第 8 章「技能自主生成（M2）」机制的产物，本章我们只关心「它们如何被拼进 prompt」，生成过程留到第 8 章。

### 6.3 跑一遍：从 Level 0 到 Level 1，再看热更新

&emsp;&emsp;光看代码不够，我们搭一个临时会话目录，亲眼看分层拼接的效果。先把三层记忆和一个技能写进去，然后做第一次拼接——此时技能还没被加载，提示词里应该只有 Level 0 的快照清单。**运行后你会看到一段带 `` 标签的完整提示词，技能部分只有清单、没有正文**。

In [17]:
# 搭一个临时会话目录，模拟三层记忆 + 一个技能（未加载）
import tempfile

demo_dir = Path(tempfile.mkdtemp(prefix="hermes_prompt_"))
(demo_dir / "SOUL.md").write_text("我是一个研究助手，回答时先给结论，再补推理过程。", encoding="utf-8")
(demo_dir / "MEMORY.md").write_text("用户偏好：简洁、中文、喜欢用表格归纳。", encoding="utf-8")

# 技能目录：一份快照清单（Level 0）+ 一个技能的完整 SKILL.md
skills_dir = demo_dir / "skills"
(skills_dir / "setup-python-project").mkdir(parents=True)
(skills_dir / "SKILLS_SNAPSHOT.md").write_text(
    "- setup-python-project: 初始化一个标准 Python 项目脚手架", encoding="utf-8")
(skills_dir / "setup-python-project" / "SKILL.md").write_text(
    "---\nname: setup-python-project\ndescription: 初始化标准 Python 项目\n---\n"
    "## 步骤\n1. 建目录结构\n2. 写 pyproject.toml\n3. 配置 CI", encoding="utf-8")

prompt_v1 = build_system_prompt_mvp(demo_dir)
print("=== 第一次拼接（技能仅 Level 0 快照，无正文）===")
print(prompt_v1)

=== 第一次拼接（技能仅 Level 0 快照，无正文）===

我是一个研究助手，回答时先给结论，再补推理过程。


用户偏好：简洁、中文、喜欢用表格归纳。


- setup-python-project: 初始化一个标准 Python 项目脚手架

## Protocol

- The user's working directory is the **workspace**. `terminal`, `python_repl`,
  `read_file`, `write_file` operate inside it.
- `MEMORY.md` is your long-term notebook. Use `write_memory` for facts that
  should survive across turns.
- `SKILLS_SNAPSHOT.md` lists available skills. If one looks relevant, call
  `read_skill('<name>')` to load its full content.
- Be concise. Cite tool results, don't paraphrase them.


&emsp;&emsp;现在我们把这个技能标记为「已加载」（写一份 `loaded_skills.json`，`level=1`），再拼一次。这一次，技能的完整 `SKILL.md` 正文会被注入。**运行后对比两次的长度，你会看到提示词明显变长了——那段多出来的就是被拉进上下文的技能全文**。

In [18]:
# 把 setup-python-project 标记为已加载（Level 1），第二次拼接：技能全文被注入
(demo_dir / "loaded_skills.json").write_text(
    json.dumps([{"name": "setup-python-project", "level": 1}]), encoding="utf-8")

prompt_v2 = build_system_prompt_mvp(demo_dir)
print("=== 第二次拼接（技能升到 Level 1，全文注入）===")
print(prompt_v2)
print(f"\nLevel 1 注入后 prompt 变长：{len(prompt_v1)} → {len(prompt_v2)} 字符")

=== 第二次拼接（技能升到 Level 1，全文注入）===

我是一个研究助手，回答时先给结论，再补推理过程。


用户偏好：简洁、中文、喜欢用表格归纳。


- setup-python-project: 初始化一个标准 Python 项目脚手架


---
name: setup-python-project
description: 初始化标准 Python 项目
---
## 步骤
1. 建目录结构
2. 写 pyproject.toml
3. 配置 CI

## Protocol

- The user's working directory is the **workspace**. `terminal`, `python_repl`,
  `read_file`, `write_file` operate inside it.
- `MEMORY.md` is your long-term notebook. Use `write_memory` for facts that
  should survive across turns.
- `SKILLS_SNAPSHOT.md` lists available skills. If one looks relevant, call
  `read_skill('<name>')` to load its full content.
- Be concise. Cite tool results, don't paraphrase them.

Level 1 注入后 prompt 变长：524 → 634 字符


&emsp;&emsp;最后看「热更新」。我们改一下 `MEMORY.md`，不重启任何东西，直接再拼一次——新写进去的记忆会立刻出现在提示词里。这正是「每轮重读文件」这个看似低效的设计换来的好处：记忆文件随时可改、改完即时生效。**运行后你会看到新增的那句记忆确实进了提示词**。

In [19]:
# 热更新演示：改 MEMORY.md，不重启，重新拼接，新记忆立即生效
(demo_dir / "MEMORY.md").write_text(
    "用户偏好：简洁、中文、喜欢用表格归纳。\n新增事实：用户正在学习 LangGraph。", encoding="utf-8")

prompt_v3 = build_system_prompt_mvp(demo_dir)
print("MEMORY 改动后是否立即进入 prompt：", "正在学习 LangGraph" in prompt_v3)

MEMORY 改动后是否立即进入 prompt： True


&emsp;&emsp;运行后如果打印出 `True`，热更新特性就验证了——改完 `MEMORY.md`、重新拼接，新内容立刻进了 `system_prompt`，全程没有重启任何服务。这看似低效的「每轮重读文件」，换来的正是「记忆随时可改、改完即生效」的透明性，也是后面第 11 章「AI 优化记忆」改完能立即反映到 Agent 行为上的前提。

## <center>第7章：Agent 内核 + 中间件钩子——自我进化的触发器</center>

&emsp;&emsp;记忆能存、能读，还能拼成系统提示词喂给模型了，但这套记忆系统始终是被动的——它只负责「存与读」，不负责「在什么时候该做什么」。整个自我进化要转起来，需要一个**触发器**：在每一轮 Agent 跑完之后，有人站出来说「刚才这一轮用了好几个工具，值得反思一下要不要提炼成技能」。在 `FuFan-OpenHermes` 里，这个触发器就是 `AgentMiddleware` 的钩子。

&emsp;&emsp;这一章是整门课第一次真实调用大模型。我们会先理解 `create_agent + AgentMiddleware` 这个组合为什么是进化机制的入口，然后定义一个最小的中间件，真跑一轮 Agent，亲眼观察钩子在什么时机被触发。本章定义的 `build_mini_llm` 大模型工厂，会被第 8、9、11、12 章反复复用，所以我们会把它写成一个干净的工厂函数。

### 7.1 配置大模型访问凭证

&emsp;&emsp;从这一章开始我们要真实调用 DeepSeek，所以先要配置 API 访问凭证。为了保护敏感信息，我们用环境变量的方式管理 API key，绝不把 key 硬编码进代码。

&emsp;&emsp;如果你还没有 `.env` 文件，需要在能被加载到的位置（本课统一从 `~/.claude/.env` 读取）创建它，内容形如下面这样。`.env` 文件包含敏感信息，切勿提交到 Git 仓库，建议在 `.gitignore` 里加上 `.env`。

In [ ]:
DEEPSEEK_API_KEY=your_actual_deepseek_api_key_here

&emsp;&emsp;首先安装本课依赖。我们把全部依赖和精确版本号写在了 `requirements.txt` 里，一条命令装齐，避免在 cell 里堆一长串包名。

In [ ]:
# 安装本课依赖（版本锁定在 requirements.txt，保证可复现）
!pip install -r requirements.txt -q

&emsp;&emsp;接下来加载环境变量。这里有一个容易被忽略但很重要的细节：我们顺手把 `LANGCHAIN_TRACING_V2` 关掉了。`LangChain` 默认会尝试把运行轨迹上报到 LangSmith，如果你没配 LangSmith 的 key，它会反复打印一长串 `403 Forbidden` 的报错噪音——这些噪音不影响功能，但会严重干扰你看真正的输出。**运行后你会看到 key 加载成功的提示；如果看到「未找到」，请检查 `.env` 文件路径和内容**。

In [23]:
# 加载环境变量 + 关闭 LangSmith 上报（避免无关的 403 噪音污染输出）
import os
from dotenv import load_dotenv

# 关闭 LangChain 默认的 LangSmith 轨迹上报（没配 key 会刷 403 报错）
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# 从 ~/.claude/.env 加载 DEEPSEEK_API_KEY
load_dotenv(os.path.expanduser("~/.claude/.env"))
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

if DEEPSEEK_API_KEY:
    print("[OK] DEEPSEEK_API_KEY 加载成功")
else:
    print("[缺失] 未找到 DEEPSEEK_API_KEY，请检查 ~/.claude/.env")

[OK] DEEPSEEK_API_KEY 加载成功


&emsp;&emsp;看到 `[OK]` 就说明凭证就绪了。下面我们把大模型的构造封装成一个工厂函数——这是本章给后续章节留下的「公共资产」。

### 7.2 build_mini_llm：全课复用的大模型工厂

&emsp;&emsp;项目源码 `agent/llm.py:7` 用一个 `build_llm` 工厂函数统一构造大模型对象，集中管理凭证和参数。我们也照这个思路写一个 `build_mini_llm`。这样做的好处是：后面每一章要用大模型时，只要调一次这个工厂，不用重复写一堆 `ChatOpenAI(...)` 配置。

&emsp;&emsp;这里有一个参数要特别留意——`streaming`。项目源码 `agent/llm.py:16` 把它做成可配置参数，是因为**结构化输出必须关掉流式**（`streaming=False`），否则 DeepSeek 端会报 Provider error；而普通对话开着流式（`streaming=True`）能更快看到首字。我们的工厂默认开流式，结构化输出的章节会显式传 `streaming=False`。

In [24]:
# 全课复用的大模型工厂（对照源码 agent/llm.py:7 的 build_llm）
from langchain_openai import ChatOpenAI

def build_mini_llm(streaming: bool = True, max_tokens: int = 1024) -> ChatOpenAI:
    """构造连接 DeepSeek 官方端点的 ChatOpenAI 对象。

    Args:
        streaming: 是否流式输出。普通对话用 True；结构化输出必须传 False。
        max_tokens: 单次回复最大 token 数，演示场景设小一点跑得快。
    Returns:
        配置好凭证和 base_url 的 ChatOpenAI 实例。
    """
    return ChatOpenAI(
        model="deepseek-v4-pro",                  # DeepSeek 官方主力模型
        api_key=DEEPSEEK_API_KEY,
        base_url="https://api.deepseek.com/v1",   # OpenAI 兼容端点
        streaming=streaming,
        max_tokens=max_tokens,
    )

# 冒烟测试：构造一个对象并确认类型
_test_llm = build_mini_llm()
print("大模型工厂就绪：", type(_test_llm).__name__)

大模型工厂就绪： ChatOpenAI


&emsp;&emsp;工厂函数就绪了。注意我们把 `max_tokens` 默认设成了 1024——这是演示场景的刻意选择，回复短一点、跑得快一点。在真实项目里这个值会大得多。接下来我们写本章的主角：中间件。

### 7.3 中间件钩子 + nudge：进化的触发器

&emsp;&emsp;现在进入本章核心。`FuFan-OpenHermes` 进化机制的入口是一个继承自 `AgentMiddleware` 的中间件类——源码里叫 `SkillEngineMiddleware`（`middleware.py:63`）。中间件提供了一组「钩子」，会在 Agent 运行的不同阶段被框架自动回调。我们关心的是 `aafter_agent` 这个钩子，它在每一轮 Agent 跑完之后触发，是「用过工具就反思」的切入点。

&emsp;&emsp;这里有一个真实的踩坑点，项目源码 `middleware.py:3-14` 的 `API DRIFT NOTE` 注释专门记录了它，我们来讲清楚。

> **【踩坑预警】**：`AgentMiddleware` 同时提供 `after_agent`（同步）和 `aafter_agent`（异步）两个钩子。当你的 Agent 走的是 `ainvoke / astream` 这种异步路径时（生产代码几乎都走异步），**必须重写 `aafter_agent` 才会被框架正确调度**；如果你只重写了同步的 `after_agent`，异步路径下它不会被调用，你的进化逻辑会「静默不触发」——代码不报错，但钩子里的逻辑一次都没跑。排查方法：在钩子里加一行 `print`，跑一轮看打印有没有出现，没出现就是重写错了钩子。源码 `middleware.py:3-14` 的 API DRIFT NOTE 里有完整记录。

&emsp;&emsp;理解了这个坑，再来看新架构的核心设计：钩子并不是每轮都干重活，而是引入了一个 **nudge 机制**——每 N 轮才触发一次后台复盘（默认 `N=5`，对照源码 `config.py:30` 的 `MEMORY_REFLECTION_INTERVAL`）。判断逻辑是 `n % interval == 0`，在源码 `run_for_session`（`middleware.py:138`）的 nudge 判断处（`middleware.py:148`）可以直接看到。这样设计的好处是：高频对话不会每轮都触发重量级复盘，系统资源的消耗可控。

&emsp;&emsp;真正到了 nudge 命中的那一轮，生产代码会用 `asyncio.create_task` 把 fork 子 agent 甩到后台「发射后不管」——这一点非常关键。钩子本身仍然被 `await`，如果在钩子里同步等待子 agent 跑完，用户拿到回复的时间就会被拖慢。`create_task` 还要传全新的 `contextvars.Context()` 来隔离父 checkpointer，否则主对话关库后子 agent 会报 "closed database"（源码 `_dispatch` 的注释 `middleware.py:114-117` 有完整说明）。下面我们先用最简版演示触发节奏，不复现 `create_task` 的完整异步隔离逻辑。

&emsp;&emsp;下面定义最小复现版的中间件，把 nudge 判断逻辑完整保留，命中时打印一行提示。

In [25]:
# 最小复现进化触发器：每轮结束判断是否到 nudge 点（对照源码 middleware.py:63 SkillEngineMiddleware / run_for_session:148）
from langchain.agents.middleware import AgentMiddleware

REFLECTION_INTERVAL = 5  # 每隔几轮触发一次后台复盘（对照源码 config.py:30 的 MEMORY_REFLECTION_INTERVAL）

class SkillEngineMiniMiddleware(AgentMiddleware):
    """进化触发器：每轮 Agent 结束后，判断是否到达 nudge 点（该 fork 后台子 agent 复盘了）。

    对照源码 middleware.py：aafter_agent 钩子（86）调度，run_for_session（138）里
    用 `n % interval == 0` 判断 nudge（148）。生产代码 nudge 命中后会 asyncio.create_task
    把 fork 子 agent 甩到后台，这里用打印演示触发时机。
    """
    def __init__(self, interval: int = REFLECTION_INTERVAL):
        super().__init__()
        self._interval = interval
        self._turns = 0
        self.nudge_log = []   # 记录哪几轮触发了 nudge，供事后检查

    async def aafter_agent(self, state, runtime):
        # state 在 LangChain 1.x 里形如 {"messages": [...]}（对照 API DRIFT NOTE，middleware.py:9）
        self._turns += 1
        n = self._turns
        nudged = (n > 0 and n % self._interval == 0)   # nudge 条件（对照源码 middleware.py:148）
        if nudged:
            self.nudge_log.append(n)
            print(f"[钩子触发] 第 {n} 轮结束 → nudge 命中（每 {self._interval} 轮），该 fork 后台子 agent 复盘了")
        else:
            print(f"[钩子触发] 第 {n} 轮结束 → 未到 nudge 点（{n} % {self._interval} != 0），跳过")
        return None   # 钩子返回 None 表示不修改 state

&emsp;&emsp;`SkillEngineMiniMiddleware` 的每一个字段都对照了生产源码：`_interval` 对应 `config.py:30` 的 `MEMORY_REFLECTION_INTERVAL`，`nudge_log` 是我们加的教学辅助字段方便事后检查，`aafter_agent` 里的 `n % self._interval == 0` 直接镜像了源码 `middleware.py:148` 的判断逻辑。现在来模拟连续 8 轮对话，看 nudge 在哪一轮命中。

&emsp;&emsp;下面这步不调用大模型，只演示触发节奏——我们手动循环调用钩子方法，传入一个空的 state 对象。

In [26]:
# 模拟 8 轮对话结束，观察 nudge 在第 5 轮命中（这一步不调用大模型，只演示触发节奏）
mw = SkillEngineMiniMiddleware()

for _ in range(8):
    await mw.aafter_agent({"messages": []}, None)

print("\nnudge 命中轮次：", mw.nudge_log, "（预期 [5]，第 10 轮才会再次命中）")

[钩子触发] 第 1 轮结束 → 未到 nudge 点（1 % 5 != 0），跳过
[钩子触发] 第 2 轮结束 → 未到 nudge 点（2 % 5 != 0），跳过
[钩子触发] 第 3 轮结束 → 未到 nudge 点（3 % 5 != 0），跳过
[钩子触发] 第 4 轮结束 → 未到 nudge 点（4 % 5 != 0），跳过
[钩子触发] 第 5 轮结束 → nudge 命中（每 5 轮），该 fork 后台子 agent 复盘了
[钩子触发] 第 6 轮结束 → 未到 nudge 点（6 % 5 != 0），跳过
[钩子触发] 第 7 轮结束 → 未到 nudge 点（7 % 5 != 0），跳过
[钩子触发] 第 8 轮结束 → 未到 nudge 点（8 % 5 != 0），跳过

nudge 命中轮次： [5] （预期 [5]，第 10 轮才会再次命中）


&emsp;&emsp;运行后你会看到 8 行 `[钩子触发]` 输出：第 1-4 轮和第 6-8 轮都打印「未到 nudge 点，跳过」，唯独第 5 轮打印「nudge 命中」；末行 `nudge 命中轮次： [5]`，符合预期。这个节奏正是生产系统里「5 轮对话才触发一次后台复盘」的设计意图——既不放过值得沉淀的对话，又不让每轮都跑重量级复盘拖慢响应。nudge 命中后该做什么？下一节来看。

### 7.4 fork 受限子 agent：让它自主决定存什么

&emsp;&emsp;nudge 命中后，旧架构的做法是「按规则分流 M2/M3 跑 structured pipeline」；新架构换了一个思路——**fork 一个受限子 agent，让它 agentic 地自主决定**存什么（源码 `background_review.spawn_review_subagent:63`，注释明写「取代原 structured-output pipeline」）。

&emsp;&emsp;这是「Agent 复盘 Agent」的核心设计：主 Agent 跑完，fork 一个小 Agent 回看刚才的对话，自己判断有没有值得记的持久偏好（调 `write_memory`）、有没有可复用的工作流值得沉淀成技能（调 `skill_manage`）。这个子 agent 有自主性——它看完对话之后自己决定调不调工具、调几次。

&emsp;&emsp;但这种自主性必须被约束在一个安全范围内，否则子 agent 乱调工具、或者无限递归 fork 新的子 agent，系统就会失控。源码 `background_review.py:6-10` 的注释明确写了 4 重约束：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>fork 子 agent 的 4 重安全约束</font></p>
<div class="center">

| 约束编号 | 约束内容 | 为什么需要 |
|---------|---------|-----------|
| ① | `recursion_limit=16` 步数上限 | 防止子 agent 陷入无限循环 |
| ② | 独立 `ainvoke`，不经父 `astream` | SSE 流天然隔离，不干扰主对话输出 |
| ③ | 工具集隔离：只给 `write_memory` + `skill_manage`，无 terminal/python/search | 防止子 agent 越权调用危险工具 |
| ④ | `middleware=[]` 防递归 | 子 agent 不再挂同一个 middleware，否则会无限 fork |

</div>

&emsp;&emsp;第 ④ 条尤为重要：如果子 agent 也挂了 `SkillEngineMiniMiddleware`，它每跑一轮也会 nudge，nudge 又 fork 新子 agent，新子 agent 又挂 middleware……这是无限递归的经典场景，`middleware=[]` 是最简单的切断手段。另外值得一提：这里子 agent 自由调工具（`tool_choice=auto`）在 DeepSeek 上跑得很顺；第 12 章会看到「强制指定 `tool_choice` function 名」在 DeepSeek thinking 模式下直接 400 的问题——同样是 tool calling，自由调和强制指定是两条不同的边界，届时会详细展开。

&emsp;&emsp;我们先构造子 agent 能用的两个工具。为了不污染工程目录，所有写入都落进临时会话目录，跑完即弃。

In [27]:
# 给子 agent 准备两个工具：写记忆 + 管理技能（对照源码 tools/memory_tool.py:52 + tools/skill_tool.py:108，教学简化版）
import tempfile
from pathlib import Path
from langchain_core.tools import tool

# 临时会话目录（跑完即弃，不污染工程）
REVIEW_SDIR = Path(tempfile.mkdtemp(prefix="ch7_review_"))
print("子 agent 的临时会话目录：", REVIEW_SDIR)

def make_review_tools(session_dir: Path):
    """构造子 agent 可用的两个工具（闭包绑定 session_dir）。"""
    @tool
    def write_memory(entry: str) -> str:
        """把一条值得长期记住的用户画像/持久偏好/协作期望写入 MEMORY.md。entry: 记忆内容。"""
        f = session_dir / "MEMORY.md"
        old = f.read_text(encoding="utf-8") if f.exists() else ""
        f.write_text(old + f"- {entry}\n", encoding="utf-8")
        return f"已写入记忆：{entry[:30]}"

    @tool
    def skill_manage(action: str, name: str, body: str = "") -> str:
        """沉淀可复用技能。action: create(新建)/patch(改进)；name: kebab-case 技能名；body: SKILL.md 正文。"""
        d = session_dir / "skills" / name
        d.mkdir(parents=True, exist_ok=True)
        (d / "SKILL.md").write_text(f"---\nname: {name}\nversion: 1.0\n---\n\n{body}", encoding="utf-8")
        return f"已 {action} 技能：{name}"

    return [write_memory, skill_manage]

子 agent 的临时会话目录： /var/folders/fl/8wq5_lz53ln9ypplts4z_1tr0000gn/T/ch7_review_xbspn2p9


&emsp;&emsp;`write_memory` 对照源码 `tools/memory_tool.py:52` 的 `create_write_memory_tool`，`skill_manage` 对照 `tools/skill_tool.py:108` 的 `create_skill_manage_tool`，这里是教学简化版，省去了真实工程里的路径管理和版本追踪逻辑，保留了核心的写入行为。接下来准备引导词和一段模拟的对话快照。

&emsp;&emsp;引导词 `COMBINED_REVIEW_PROMPT` 在源码 `background_review.py:27` 里定义，它告诉子 agent 要从两个维度复盘：记忆维度（持久偏好）和技能维度（可复用工作流）；如果都没有，就回「Nothing to save.」停止。

In [28]:
# 复盘引导词（移植源码 background_review.py:27 的 COMBINED_REVIEW_PROMPT，精简）
COMBINED_REVIEW_PROMPT = """你是后台自审 agent，复盘刚结束的一轮对话，自主决定是否沉淀「记忆」或「技能」。
你只能调用：write_memory / skill_manage 两个工具。
- 记忆维度：对话里有没有值得长期记住的 user 画像 / 持久偏好 / 协作期望？有就 write_memory 存。
- 技能维度：这轮有没有可复用的工作流 / 技巧值得沉淀成技能？有就 skill_manage(action=create, name=kebab-case, body=完整SKILL.md正文)。
两个维度可以都做。若都没有，回复「Nothing to save.」并停止。
现在复盘下面的对话，自主调工具沉淀，完成后简述你做了什么。
"""

# 一段刚结束的对话快照：用户既透露了持久偏好，又教了一套可复用工作流
review_trace = """USER: 我每次部署 Docker 容器都忘了配健康检查，容器假死了都发现不了。
ASSISTANT: 给你一套标准流程：① Dockerfile 加 HEALTHCHECK 指令配探测命令 ② docker-compose 配 healthcheck 的 interval/timeout/retries ③ 配 depends_on condition: service_healthy 控制启动顺序 ④ 用 docker ps 看 STATUS 的 (healthy) 标记验证。
USER: 太实用了，这套流程我以后每次都要用。另外记住我们团队统一用 docker-compose v2 语法，不要给我 v1 的写法。"""

&emsp;&emsp;对话快照里藏了两种值得沉淀的信息：用户说「记住我们团队统一用 docker-compose v2 语法」——这是持久偏好，应该 `write_memory`；用户说「这套流程我以后每次都要用」——这是可复用工作流，应该 `skill_manage:create`。我们来看子 agent 自己能不能发现并分别处理。

In [29]:
# fork 受限子 agent 复盘这段对话，看它自主调了哪些工具（对照源码 background_review.spawn_review_subagent:63）
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 4 重约束：③ 工具集隔离（只 2 个工具）+ ④ middleware=[] 防递归 + ① recursion_limit 步数上限
review_agent = create_agent(
    model=build_mini_llm(streaming=False, max_tokens=2048),   # 复用 7.2 的工厂；结构化/agentic 关流式
    tools=make_review_tools(REVIEW_SDIR),
    middleware=[],                                            # ④ 子 agent 不再挂 middleware，防无限 fork
).with_config(recursion_limit=16)                            # ① 步数上限

prompt = COMBINED_REVIEW_PROMPT + "\n\n--- 本轮对话 ---\n" + review_trace

# ② 独立 ainvoke（不经父对话 astream）；config={"configurable": {}} 隔离父 checkpointer
result = await review_agent.ainvoke(
    {"messages": [HumanMessage(content=prompt)]},
    config={"configurable": {}},
)

# 从 result.messages 的 tool_calls 提取子 agent 做了哪些动作（对照源码 _extract_tool_actions:116）
actions = []
for m in result["messages"]:
    for tc in getattr(m, "tool_calls", []) or []:
        if tc["name"] in ("write_memory", "skill_manage"):
            val = tc["args"].get("action") or tc["args"].get("name") or tc["args"].get("entry") or ""
            actions.append(f"{tc['name']}:{str(val)[:30]}")

print("子 agent 自主调用的工具动作：", actions)
print("\n子 agent 复盘总结：", result["messages"][-1].content[:150])

子 agent 自主调用的工具动作： ['write_memory:用户团队统一使用 docker-compose v2 语法，', 'skill_manage:create']

子 agent 复盘总结： 完成。沉淀了两项内容：

**记忆** — 记录了用户的持久偏好：团队统一使用 docker-compose v2 语法（不写 v1），且每次部署 Docker 容器都必须配置健康检查。

**技能** `docker-healthcheck-setup` — 将四步标准流程沉淀为可复用技能，包含 


&emsp;&emsp;运行后你会看到类似这样的输出：`actions` 列表里有 `write_memory:` 加上团队用 v2 语法那条偏好，以及 `skill_manage:docker-healthcheck-workflow` 之类的技能名；最后一行是子 agent 自己总结的一句话，说明它做了什么。这是 agentic 输出，每次措辞略有不同，但调用的工具种类应该一致（记忆 + 技能都覆盖到）。我们再看一眼临时目录里真实落盘的产物。

In [30]:
# 看子 agent 真的往临时目录里沉淀了什么
mem = REVIEW_SDIR / "MEMORY.md"
if mem.exists():
    print("=== 子 agent 写入的 MEMORY.md ===")
    print(mem.read_text(encoding="utf-8"))

for sk in (REVIEW_SDIR / "skills").glob("*/SKILL.md") if (REVIEW_SDIR / "skills").exists() else []:
    print(f"=== 子 agent 创建的技能 {sk.parent.name}/SKILL.md ===")
    print(sk.read_text(encoding="utf-8")[:300])

=== 子 agent 写入的 MEMORY.md ===
- 用户团队统一使用 docker-compose v2 语法，不要 v1 写法。每次部署 Docker 容器都必须配置健康检查（healthcheck），避免容器假死无法发现。

=== 子 agent 创建的技能 docker-healthcheck-setup/SKILL.md ===
---
name: docker-healthcheck-setup
version: 1.0
---

# Docker 容器健康检查配置流程

每次部署 Docker 容器时，按以下标准流程配置健康检查，防止容器假死。

## 适用场景
- 任何 Dockerfile 构建或 docker-compose 部署
- 团队统一使用 docker-compose v2 语法

## 标准流程

### ① Dockerfile：添加 HEALTHCHECK 指令
```dockerfile
HEALTHCHECK --interval=30s --timeout=3s --start-peri


&emsp;&emsp;你会看到类似这样的结果：`MEMORY.md` 里有一条关于「团队统一用 docker-compose v2 语法」的持久偏好记录；`skills/docker-healthcheck-workflow/SKILL.md`（或类似名称）是一份带四步流程的完整技能文档。这两个文件都是子 agent 在没有任何规则约束的情况下，**自主决定要存、自主调工具写入**的——这正是「agentic 复盘」和「structured pipeline」的本质区别：前者让 agent 自己判断值不值得存、存什么，后者是按预设规则机械分类。

## <center>第8章：技能自主生成</center>

&emsp;&emsp;我们终于走到进化闭环的第一个核心：Document——让 Agent 把用过的工作流，自己提炼成一份技能说明书。这一章我们要回答的问题是：当我们想把一段「用过工具完成任务」的对话沉淀成一份可复用技能时，怎么让大模型判断这套工作流值不值得提炼？又怎么让它把技能写成结构化的文件？上一章里，后台子 agent 是通过 `skill_manage` 工具自主沉淀技能的；这一章我们拆开看技能生成本身的引擎 `generator.py`——它既被前端 `/skills` 页的「从这轮对话生成技能」显式调用，也是子 agent 沉淀新技能时背后的同款逻辑。

&emsp;&emsp;这一章还藏着整门课最有复用价值的一个技术点——`json_mode + PydanticOutputParser` 的结构化输出范式。它不只在 `FuFan-OpenHermes` 里用，你可以把它直接搬到任何需要「让大模型返回严格结构化数据」的项目里。我们会把这个范式提炼成一个独立的封装，后面第 9、11、12 章都会复用它，所以这一章请你重点掌握。本章的大模型对象复用第 7 章定义的 `build_mini_llm`。

### 8.1 技能自主生成 的两步：判定 → 写入

&emsp;&emsp;我们先把 技能自主生成的完整流程拆成两步，建立一个整体框架，再逐步动手实现。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>技能自主生成的两步</font></p>
<div class="center">

| 步骤 | 做什么 | 源码锚点 |
|------|--------|---------|
| 判定 + 生成 | 把对话整理成文本，让大模型判断「值不值得提炼」并生成技能 | `skill_engine/generator.py:62,115` |
| 写入 | 解决命名冲突后写入 `SKILL.md` | `skill_engine/generator.py:82,200` |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612165956040.png" width=50%></div>

&emsp;&emsp;第一步「判定 + 生成」是核心——把这段对话整理成一段文本喂给大模型，让它判断「这套工作流是不是真的可复用」，可复用就同时生成一份结构化技能（不可复用则判 `NO_SKILL` 丢弃）。第二步「写入」把生成的技能解决命名冲突后落盘成 `SKILL.md`。我们逐步实现，重点在第一步的结构化输出范式。

### 8.2 准备一段对话轨迹

&emsp;&emsp;技能自主生成 的输入，是一段「用过工具完成任务」的对话轨迹。源码 `generator.py:62` 的 `_format_turn_for_generator` 会把这样一轮对话整理成一段纯文本，再喂给大模型判定。我们先构造一个模拟「搭 Python 项目脚手架」的对话轨迹（`fake_trace`）当作输入。

&emsp;&emsp;下面这段构造一个用了 4 个工具的假对话轨迹。**运行后你会看到这段对话的结构**——它会成为下一节判定阶段的输入。

In [32]:
# 构造 技能自主生成 的输入：一段"用过工具完成任务"的对话轨迹
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

# 模拟一段"用了 4 个工具搭脚手架"的假对话轨迹（技能自主生成 的输入）
fake_trace = [
    HumanMessage("帮我搭一个标准的 Python 项目脚手架"),
    AIMessage(content="好的，我来执行。", tool_calls=[
        {"name": "terminal", "args": {"command": "git init"}, "id": "c1"},
        {"name": "write_file", "args": {"path": "requirements.txt"}, "id": "c2"},
        {"name": "terminal", "args": {"command": "mkdir -p src tests"}, "id": "c3"},
        {"name": "write_file", "args": {"path": "README.md"}, "id": "c4"},
    ]),
    ToolMessage(content="done", tool_call_id="c1"),
]

# 统计这轮用了几个工具（仅用于展示轨迹结构）
_n_tools = sum(len(getattr(m, "tool_calls", []) or []) for m in fake_trace)
print(f"对话轨迹构造完成：{len(fake_trace)} 条消息，含 {_n_tools} 次工具调用")

对话轨迹构造完成：3 条消息，含 4 次工具调用


&emsp;&emsp;这段对话用了 4 个工具完成了「搭脚手架」这个任务，是一个典型的「可能值得提炼成技能」的候选。注意 `AIMessage` 的 `tool_calls` 是一个字典列表，每个字典有 `name / args / id` 三个字段——这是 `LangChain` 工具调用的标准格式，记住它，后面构造测试数据都用得上。

### 8.3 判定的核心：json_mode 结构化输出范式

&emsp;&emsp;现在进入这一章、也是整门课最关键的技术点。检测通过后，我们要让大模型判断「这套工作流值不值得提炼」，并且——如果值得——直接生成一份结构化的技能。难点在于：**怎么保证大模型返回的是我们能直接用的结构化数据，而不是一段需要再解析的自由文本？**

&emsp;&emsp;`function_calling` 是官方推荐的结构化输出方式，直觉上最稳。但实测结论相反：**在被降级到备用 provider 的场景下，`function_calling` 反而没有 `json_mode` 稳**。这不是我们凭空猜的，是项目源码 `skill_engine/generator.py:130-131` 的注释里记录的实测结论。我们把它原文摘出来对照。

> **【关于 json_mode vs function_calling】**：项目源码 `generator.py:130-131` 的注释里记录了一组实测数字——`json_object` 模式（即 LangChain `with_structured_output(method="json_mode")` 底层对应的 OpenAI/DeepSeek `response_format={"type":"json_object"}`，本课这两个名字指同一个模式）**5/5 成功**，而 `function_calling` 模式 **4/5 成功**（1 次失败具体是「zero valid function call」或字段返回字符串导致 Pydantic 校验失败）。这两组数字来自项目作者在小样本下的非正式观察，不是独立 benchmark——它说明的是**在项目实际使用的 OpenRouter 路径上** 两者都基本能跑、但 `json_object` 略稳。考虑到 `json_mode` 不自动注入 schema 需要手动用 `PydanticOutputParser` 拼到 prompt 末尾这点额外成本，项目最终在技能生成、技能强化、记忆优化这三处都选了 `json_mode`。注意：到第 12 章我们会用四种组合的 curl 实测展示另一个更硬的边界（DeepSeek thinking 模式下 `tool_choice` 强制指定直接 400），到时你再回头看本节会理解得更完整。

&emsp;&emsp;`json_mode` 有一个使用上的关键细节必须讲清楚，否则你照着写一定会踩坑。

> **【踩坑预警】**：`json_mode` **不会自动把数据结构（schema）注入到 prompt 里**——它只是告诉大模型「请返回合法 JSON」，但大模型并不知道这个 JSON 该长什么样。所以你必须手动用 `PydanticOutputParser(...).get_format_instructions()` 把 schema 描述拼到 prompt 末尾（对照源码 `generator.py:141`）。如果你漏了这一步，大模型会返回一个合法但字段完全不对的 JSON，然后 Pydantic 校验失败。这是 `json_mode` 和 `function_calling` 最大的使用差异——后者会自动带上 schema，前者要你手动注入。

&emsp;&emsp;讲清楚了原理，我们就来定义技能的数据契约。对照源码 `skill_engine/schemas.py`，技能本身是 `GeneratedSkill`（这里简化为 `MiniSkill`），顶层判定结果是 `GeneratorOutput`——它要么是「生成 + 技能内容」，要么是「不生成」。

In [33]:
# 判定步（1/3）：定义结构化输出的数据契约（对照源码 skill_engine/schemas.py）
from typing import Literal, Optional
from pydantic import BaseModel, Field

class MiniSkill(BaseModel):
    """生成的技能，对应 SKILL.md 的核心字段（对照 schemas.py:9 GeneratedSkill）。"""
    name: str = Field(description="kebab-case 技能名，小写字母开头")
    description: str = Field(description="一句话说明：是什么 + 何时用")
    triggers: list[str] = Field(description="触发该技能的关键词或短语")

class GeneratorOutput(BaseModel):
    """顶层判定结果：要么 GENERATE+技能，要么 NO_SKILL（对照 schemas.py:45）。"""
    decision: Literal["GENERATE", "NO_SKILL"]  # 枚举：只能是这两个值之一
    reasoning: str = Field(description="一句话说明做出该判定的理由")
    skill: Optional[MiniSkill] = None  # 仅 GENERATE 时存在

print("数据契约就绪：GeneratorOutput（decision / reasoning / skill）")

数据契约就绪：GeneratorOutput（decision / reasoning / skill）


&emsp;&emsp;数据契约定义好了。注意 `decision` 用的是 `Literal["GENERATE", "NO_SKILL"]`——这是一个枚举约束，大模型只能返回这两个值之一，返回别的会被 Pydantic 拦下。这种「用类型系统约束大模型输出」的思路，正是结构化输出范式的核心。下面我们把这个范式封装成一个可复用的函数，这是本章留给后续章节的核心资产。

In [34]:
# 判定步（2/3）：封装 json_mode 结构化输出范式（后续 9/11/12 章复用，不再重复讲）
from langchain_core.output_parsers import PydanticOutputParser

async def structured_call(schema_cls, prompt_body: str):
    """json_mode 结构化输出的标准范式（对照源码 generator.py:115-145）。

    Args:
        schema_cls: 期望输出的 Pydantic 模型类。
        prompt_body: 业务 prompt 正文（不含 schema 说明，本函数会自动拼）。
    Returns:
        schema_cls 的实例（已通过 Pydantic 校验）。
    """
    # 关键 1：结构化输出必须 streaming=False，否则 DeepSeek 端报 Provider error
    llm = build_mini_llm(streaming=False, max_tokens=4096)
    # 关键 2：用 method="json_mode"，比 function_calling 在降级场景更稳（实测 5/5 vs 4/5）
    structured = llm.with_structured_output(schema_cls, method="json_mode")
    # 关键 3：json_mode 不自动注入 schema，必须手动把格式说明拼到 prompt 末尾
    schema_hint = PydanticOutputParser(pydantic_object=schema_cls).get_format_instructions()
    full_prompt = prompt_body + "\n\n" + schema_hint
    # 异步调用，返回已校验的 Pydantic 实例
    return await structured.ainvoke(full_prompt)

print("json_mode 范式已封装为 structured_call()，后续章节直接复用")

json_mode 范式已封装为 structured_call()，后续章节直接复用


&emsp;&emsp;这个 `structured_call` 就是整门课的「公共结构化输出引擎」。它把三个关键点固化了下来：`streaming=False`（否则报错）、`method="json_mode"`（更稳）、手动注入 `schema_hint`（否则字段不对）。后面任何一章要让大模型返回结构化数据，只需要传一个 Pydantic 类和一段业务 prompt，剩下的它全包了。记住这三个关键点，你在自己项目里也能立刻复用。

### 8.4 端到端：让大模型生成一个技能

&emsp;&emsp;范式封装好了，我们就用它真跑一次完整的 技能自主生成。我们把上面的 `fake_trace` 整理成文本喂给大模型，让它判断「搭 Python 项目脚手架」这套工作流值不值得提炼。**运行后你会看到大模型的判定结果**：`decision` 是 `GENERATE` 还是 `NO_SKILL`，以及它的理由；如果是 `GENERATE`，还会打印出它生成的技能名和描述。

In [35]:
# 判定步（3/3）：端到端跑一次 技能自主生成 判定 + 生成（复用 structured_call 范式）
GENERATOR_SYSTEM_PROMPT = (
    "你是一个技能提炼器。下面是一段 Agent 刚完成的对话轨迹。"
    "请判断这套工作流是否值得提炼成一份可复用的技能（SKILL.md）。"
    "可复用的工作流（如项目脚手架、代码审查流程）判 GENERATE；"
    "一次性的简单操作（如只是 ls 一下目录）判 NO_SKILL。"
)

def format_trace(messages) -> str:
    """把消息列表整理成可读文本喂给大模型。"""
    lines = []
    for m in messages:
        role = type(m).__name__
        tcs = getattr(m, "tool_calls", []) or []
        lines.append(f"{role}: {m.content}")
        for tc in tcs:
            lines.append(f"  调用工具 {tc['name']}({tc['args']})")
    return "\n".join(lines)

# 拼业务 prompt 并调用范式
prompt_body = GENERATOR_SYSTEM_PROMPT + "\n\n--- 对话轨迹 ---\n" + format_trace(fake_trace)
result = await structured_call(GeneratorOutput, prompt_body)

print("判定结果：", result.decision)
print("判定理由：", result.reasoning)
if result.decision == "GENERATE" and result.skill:
    print("生成的技能名：", result.skill.name)
    print("技能描述：", result.skill.description)
    print("触发词：", result.skill.triggers)

判定结果： GENERATE
判定理由： 搭建Python项目脚手架是可复用的标准化流程，适合提炼为技能。
生成的技能名： python-project-scaffold
技能描述： 快速搭建一个标准的Python项目脚手架，包含git初始化、目录结构、requirements.txt和README。
触发词： ['搭一个标准的 Python 项目脚手架', '创建Python项目骨架', '初始化Python项目', '搭建Python项目']


&emsp;&emsp;运行后你大概率会看到 `decision` 是 `GENERATE`——因为「搭 Python 项目脚手架」确实是个高度可复用的工作流。大模型会给它起一个 kebab-case 的名字（比如 `python-project-scaffold`），并提取出触发词。这就是 M2 的核心链路：把对话整理成文本 → 大模型判定值得 → 生成结构化技能。下面我们再看一个反例，验证大模型是有判断力的。

### 8.5 验证大模型的判断力：简单任务会被拒绝

&emsp;&emsp;一个好的技能生成器，不应该「只要用了工具就生成技能」，否则会产生一堆没价值的垃圾技能。我们构造一个反例：一个只是「ls 一下目录」的简单任务，看大模型会不会明智地判 `NO_SKILL`。这是一个有无对比——前面是「值得提炼」的正例，这里是「不值得提炼」的反例。

&emsp;&emsp;**运行后你会看到，对于这种一次性的琐碎操作，大模型大概率判 `NO_SKILL`**，证明它确实在「判断价值」而不是「机械生成」。

In [36]:
# 反例验证：简单任务应被判 NO_SKILL（与上面的正例形成对比）
simple_trace = [
    HumanMessage("看一下当前目录有哪些文件"),
    AIMessage(content="好的", tool_calls=[
        {"name": "terminal", "args": {"command": "ls -la"}, "id": "s1"},
    ]),
    ToolMessage(content="file1.txt file2.txt", tool_call_id="s1"),
]

prompt_body2 = GENERATOR_SYSTEM_PROMPT + "\n\n--- 对话轨迹 ---\n" + format_trace(simple_trace)
result2 = await structured_call(GeneratorOutput, prompt_body2)
print("简单任务的判定：", result2.decision)
print("判定理由：", result2.reasoning)

简单任务的判定： NO_SKILL
判定理由： 简单的目录列表查询，一次性操作，无复用价值。


&emsp;&emsp;对比两次结果你会发现：复杂可复用的工作流被判 `GENERATE`，一次性的琐碎操作被判 `NO_SKILL`。这种「价值判断」正是让自我进化不至于产生噪音的关键——Agent 不是把每件事都记下来，而是只沉淀真正值得复用的工作流。

### 8.6 本章验证：两层验证

&emsp;&emsp;本章验证分两层：Tier 1 检查不依赖大模型的纯结构部分（数据契约约束），Tier 2 验证完整的大模型生成链路。先看 Tier 1。

In [37]:
# Tier 1（组件级）：把数据契约约束逐项打印出来看（不联网）
from pydantic import ValidationError

# 检查 1：GeneratorOutput 能正常构造合法实例
ok = GeneratorOutput(decision="GENERATE", reasoning="可复用工作流",
                     skill=MiniSkill(name="setup-py", description="搭建 Python 项目脚手架的标准流程",
                                     triggers=["脚手架", "项目初始化"]))
print("合法实例构造成功，decision =", ok.decision, "，skill 非空 =", ok.skill is not None)

# 检查 2：decision 只能是枚举值，传非法值会被 Pydantic 拦下——把拦截报错打印出来看
try:
    GeneratorOutput(decision="MAYBE", reasoning="x")
    print("非法 decision 'MAYBE' 未被拦截（异常！）")
    _blocked = False
except ValidationError as e:
    _blocked = True
    print("非法 decision 'MAYBE' 被 Pydantic 拦下，报错：", e.errors()[0]["msg"])

# 机器门控哨兵
assert ok.skill is not None and _blocked
print("\n[PASS] 数据契约对非法值有约束")

合法实例构造成功，decision = GENERATE ，skill 非空 = True
非法 decision 'MAYBE' 被 Pydantic 拦下，报错： Input should be 'GENERATE' or 'NO_SKILL'

[PASS] 数据契约对非法值有约束


&emsp;&emsp;Tier 1 确认了零件正确。Tier 2 我们验证完整的大模型生成链路——真跑一次，把模型返回的判定与技能名格式打印出来——若是 `GENERATE`，技能名应符合 kebab-case 格式。**运行后你会看到模型的判定结果与生成的技能名**。

In [ ]:
# Tier 2（端到端）：真跑一次 技能自主生成，把模型返回的结构与格式打印出来看
import re

t2_result = await structured_call(GeneratorOutput, prompt_body)  # 复用上面的脚手架 prompt
print("模型判定 decision =", t2_result.decision, "（合法枚举：GENERATE / NO_SKILL）")
if t2_result.decision == "GENERATE":
    _name = t2_result.skill.name
    _kebab = bool(re.fullmatch(r"[a-z][a-z0-9-]+", _name))
    print("生成的技能名 =", _name)
    print("技能名是合法 kebab-case =", _kebab)
    # 机器门控哨兵：GENERATE 时技能名必须合法
    assert t2_result.skill is not None and _kebab
    print(f"\n[PASS] 端到端生成技能：{_name}")
else:
    # 机器门控哨兵：decision 必须落在合法枚举内
    assert t2_result.decision in ["GENERATE", "NO_SKILL"]
    print(f"\n[PASS] 端到端判定为 {t2_result.decision}（也是合法结果）")

模型判定 decision = GENERATE （合法枚举：GENERATE / NO_SKILL）
生成的技能名 = python-scaffold
技能名是合法 kebab-case = True

[PASS] 端到端生成技能：python-scaffold


&emsp;&emsp;两层验证通过，我们就完整复现了 技能自主生成——Agent 能在用过工具后，自己判断并生成技能文档。进化闭环的 Document 这一步就有了。但技能写完不是终点：下次再用这个技能时，如果发现它有不足，能不能让 Agent 自己改进它？这正是下一章，进化闭环的 Improve。

## <center>第9章：技能自主强化</center>

&emsp;&emsp;前两章我们让 Agent 学会了「从零生成技能」（第8章 M2）。但技能第一版往往不完美——本章看技能怎么**主动变强**。当前项目的做法不是"等用过之后被动反思"，而是 `evolver.py` 的**多轮定向训练**（源码 `evolver.py:1` 标注 Verbal RL）：给技能配一组训练任务，让它反复"做题→自评→改规则→再做题"，像学生刷题一样在任务上把自己练到达标。

&emsp;&emsp;这一章我们复现这个训练循环，亲眼看技能的 pass_rate 从 0% 爬到 100%、版本从 v1.0 迭代到 v1.2。大模型对象继续复用第7章的 `build_mini_llm`，结构化输出走和第8章相同的 `json_mode` 范式。需要说明的是：标题里的 M3 是本课「生成→强化」教学编号，源码 `evolver.py` 把这个机制标注为 M4 Verbal RL，二者描述同一机制，不需纠结编号。

### 9.1 从「复用后反思」到「定向训练」

&emsp;&emsp;一个技能怎么变强？有两条思路值得对比。**第一条：被动**——等技能被用过几次后，回看用得好不好再改，依赖真实运行轨迹触发。**第二条：主动**——直接给技能一组代表性任务，让它当场做、自己评分、针对失败改规则，多轮迭代（`evolver.py` 的做法）。后者叫 **Verbal RL（语言强化学习）**：不调模型参数，纯靠"自然语言反思"驱动技能文档迭代；强化信号是 pass_rate，策略更新是改 SKILL.md 的规则。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>「复用后反思」vs「定向训练」两种思路对比</font></p>
<div class="center">

| 维度 | 复用后反思（被动） | 定向训练（主动，evolver.py） |
|------|-------------------|------------------------------|
| 触发方式 | 等待真实运行轨迹 | 主动给定代表性任务集 |
| 迭代信号 | 运行成功/失败感知 | pass_rate 数值（0%→100%） |
| 适用场景 | 积累足够真实案例后 | 技能刚写好、需要快速验证收敛 |

</div>

&emsp;&emsp;训练循环对照源码 `evolver.py:104` 的 for 循环，每轮分两个角色。**Actor + Evaluator**：用当前 SKILL.md 当指南做题 + 自评 pass/fail + 写反思指出规则缺口。**Curator**：根据反思增删规则、产出新版 SKILL.md + pass_rate。pass_rate 达到目标则收敛，否则进下一轮。接下来我们先把这个循环的数据契约定义清楚，再组装成完整训练。

### 9.2 数据契约：训练循环的输入输出

&emsp;&emsp;在动手写训练循环之前，先把它用到的数据结构全部定义好（对照 `skill_engine/evolver_schemas.py`）。有一个 Python 保留字坑必须提前讲清楚。

> **【踩坑预警】**：`pass` 是 Python 的保留关键字，不能直接当字段名写 `pass: bool`（语法错误）。源码 `evolver_schemas.py:25` 的解法是：字段名用 `passed`，再用 `Field(alias="pass")` 绑定别名，并开 `model_config = ConfigDict(populate_by_name=True)`，这样既能用 `passed=...` 在 Python 里构造，又能让大模型按 json_mode 输出键名 `"pass"`。如果你直接写 `pass: bool`，Python 解释器会在定义类时就报 `SyntaxError`。

&emsp;&emsp;下面把训练循环涉及的全部 schema 在一个 cell 里定义完毕。代码逐字对照源码 `evolver_schemas.py`，注释里标注了各类的源码行号。

In [39]:
# 训练循环的数据契约（对照源码 skill_engine/evolver_schemas.py）
from pydantic import BaseModel, ConfigDict, Field

class TrainingTask(BaseModel):
    """一道训练任务（对照源码 evolver_schemas.py:9）。"""
    id: str
    title: str
    prompt: str

class TaskResult(BaseModel):
    """单个任务的评估结果（对照源码 evolver_schemas.py:15）。

    'pass' 是 Python 保留字 → 字段名用 passed，alias 绑 "pass"，json_mode 让模型输出键名 "pass"。
    """
    model_config = ConfigDict(populate_by_name=True)
    task_id: str
    output: str
    passed: bool = Field(..., alias="pass")     # 对照源码 evolver_schemas.py:25

class ActorOutput(BaseModel):
    """Actor 的输出：每个任务的结果 + 一段反思（对照源码 evolver_schemas.py:28）。"""
    task_results: list[TaskResult] = Field(..., min_length=1)
    reflection: str = Field(..., min_length=10, max_length=800)

class Proposal(BaseModel):
    """Curator 提的最小修改提案（对照源码 evolver_schemas.py:33，added≤3 / removed≤2）。"""
    added: list[str] = Field(default_factory=list, max_length=3)
    removed: list[str] = Field(default_factory=list, max_length=2)
    rationale: str = Field(..., min_length=5, max_length=400)

class CuratorOutput(BaseModel):
    """Curator 的输出：提案 + 完整新版 SKILL.md + 通过率（对照源码 evolver_schemas.py:39）。"""
    proposal: Proposal
    new_skill_md: str = Field(..., min_length=40)
    pass_rate: float = Field(..., ge=0, le=1)

print("数据契约就绪：TrainingTask / TaskResult / ActorOutput / Proposal / CuratorOutput")

数据契约就绪：TrainingTask / TaskResult / ActorOutput / Proposal / CuratorOutput


&emsp;&emsp;这五个类覆盖了训练循环的全部 I/O。注意 `Proposal` 限制了 added≤3、removed≤2，这是源码的刻意约束，防止 Curator 每轮都大改规则导致技能不稳定。下面再准备训练场景——一个故意留缺陷的初始技能、一组训练任务，以及一份严格判定标准。

In [40]:
# 准备训练场景：一个故意留缺陷的初始技能 + 一组训练任务 + 判定标准
skill_md_v1 = """---
name: write-commit-message
version: 1.0
---
# 写 Git Commit Message

## 规则
- 把这次改了什么写清楚。
"""

# 判定标准的核心是「团队私有模块代号」——通用大模型无法凭常识猜到的项目私有约定，v1.0 完全没有，让第 1 轮必然 fail
pass_criteria = ("必须严格满足全部三条才算 pass：① 标题格式为 `[模块代号] 简短描述`，其中【模块代号】"
                 "必须取自团队《模块代号表》（团队私有约定，绝不是 login/auth/cache/product 这类通用英文词）"
                 "② 标题≤50字符 ③ 正文说明改动原因。任一不满足即 fail。"
                 "关键：模块代号表只可能来自当前 SKILL.md——若 SKILL.md 没有给出模块代号表，"
                 "你无从得知正确代号，必须如实判 fail，不得自行编造代号、也不得用通用英文词凑。")

task_set = [
    TrainingTask(id="t1", title="修复登录超时", prompt="你修复了登录接口因数据库连接池耗尽导致的超时 bug。"),
    TrainingTask(id="t2", title="加缓存", prompt="你给商品详情接口加了 Redis 缓存，响应从 200ms 降到 20ms。"),
]
print(f"初始技能 v1.0（只有 1 条空泛规则）+ {len(task_set)} 道训练任务 + 严格判定标准 就绪")

初始技能 v1.0（只有 1 条空泛规则）+ 2 道训练任务 + 严格判定标准 就绪


&emsp;&emsp;初始技能故意只有一条空泛规则"把改了什么写清楚"，判定标准却要求标题带上团队私有的《模块代号表》里的代号（v1.0 完全没有，而且这种项目私有约定通用大模型再强也猜不到、没法靠常识补全）。这意味着第1轮 Actor 无从得知正确代号、必然全 fail——这样才能看到训练把它"教会"的完整过程。接下来定义 Actor 和 Curator 这两个训练角色。

### 9.3 Actor + Evaluator：用技能做题并自评

&emsp;&emsp;Actor 的职责是：把当前 SKILL.md 当指南，顺序完成每个任务，再按 `pass_criteria` 客观判 pass/fail，最后写一段反思**指出具体哪条规则不够用**。它对照源码 `evolver.py:29` 的 `_actor_system_prompt` + `evolver.py:108` 的 Actor 调用。结构化输出走 `with_structured_output(method="json_mode")`——和第8章 M2 相同的 json_mode 范式。注意 `get_format_instructions()` 把 schema 拼进 prompt（json_mode 不自动注入 schema，需要手动拼），这一点和第8章一致。

In [41]:
# Actor + Evaluator：用 SKILL.md 当指南做题 + 自评 + 反思（对照源码 evolver.py:29 _actor_system_prompt / evolver.py:108 调用）
from langchain_core.output_parsers import PydanticOutputParser

ACTOR_SYS = """你是 Skill Evolver 的 Actor + Evaluator。
1. 严格只用给定 SKILL.md 的规则当指南完成每个 task（不要用 SKILL.md 之外的常识补格式），output≤80字。
2. 按此标准客观判定 pass/fail：{crit}
3. 写中文 reflection(50-200字)，指出具体哪条规则缺失导致 fail，不要泛泛说"需改进"。
严格 JSON 输出。task_results 每项含 task_id、output、pass（布尔，键名必须是 "pass"）；reflection 是字符串。"""

actor_parser = PydanticOutputParser(pydantic_object=ActorOutput)

async def call_actor(llm, skill_md, n):
    """跑一轮 Actor：用第 n 轮的 skill_md 做题并自评。返回 ActorOutput。"""
    runnable = llm.with_structured_output(ActorOutput, method="json_mode")   # 对照源码 evolver.py:108
    tasks = "\n".join(f"[{t.id}] {t.title}: {t.prompt}" for t in task_set)
    prompt = (ACTOR_SYS.format(crit=pass_criteria) +
              f"\n\n当前 SKILL.md（第{n}轮）：\n<skill>\n{skill_md}\n</skill>\n\n任务集：\n{tasks}\n\n"
              + actor_parser.get_format_instructions())
    out = await runnable.ainvoke(prompt)
    return out if isinstance(out, ActorOutput) else ActorOutput.model_validate(out)

print("Actor 调用函数 call_actor 就绪")

Actor 调用函数 call_actor 就绪


&emsp;&emsp;Actor 的演示不单独跑——放到 9.5 完整训练循环里一次跑全，可以完整看到每轮反思如何推动规则迭代，避免重复调用 LLM。接下来定义 Curator。

### 9.4 Curator：根据反思增删规则

&emsp;&emsp;Curator 读 Actor 的反思，提一个**最小修改提案**（加规则≤3 / 删规则≤2，对照源码 `evolver.py:45` 的 Curator system prompt + `evolver.py:143` 调用），并输出完整的新版 SKILL.md + pass_rate。版本号用 `bump_minor_version` 从 1.0 升到 1.1（对照源码 `skill_engine/diff.py` 的 `bump_minor_version`）。Curator 的约束是"最小修改"——不允许每轮把整个技能推倒重写，只针对反思指出的缺口做定向增删。

In [42]:
# Curator：根据反思增删规则、产出新版 SKILL.md（对照源码 evolver.py:45 Curator prompt / evolver.py:143 调用）
CURATOR_SYS = """你是 Skill Evolver 的 Curator。基于本轮 reflection 提出对 SKILL.md 的最小修改，输出修改后完整 new_skill_md。
若 reflection 指出缺《模块代号表》，必须在 new_skill_md 里列出每个涉及模块的具体私有代号（如 登录→M07、商品详情→M12），禁止用 general/default/通用英文词 之类的兜底占位代号。
约束：proposal.added≤3条(祈使句规则正文)、proposal.removed≤2条(必须是现有原句)、总规则≤14；
new_skill_md 完整覆盖整份文件，frontmatter version 必须是 {ver}，以 --- 开头；pass_rate 与 task_results 通过比例一致；rationale 50-200字。
严格 JSON 输出，中文。"""

curator_parser = PydanticOutputParser(pydantic_object=CuratorOutput)

def bump_minor_version(v):
    """次版本号 +1（对照源码 skill_engine/diff.py 的 bump_minor_version）：1.0 → 1.1。"""
    a, b = v.split("."); return f"{a}.{int(b)+1}"

async def call_curator(llm, skill_md, actor_out, next_ver):
    """跑一轮 Curator：根据 Actor 反思提 patch、产出新版。返回 CuratorOutput。"""
    runnable = llm.with_structured_output(CuratorOutput, method="json_mode")   # 对照源码 evolver.py:143
    trlines = "\n".join(f"- [{r.task_id}] {'通过' if r.passed else '未通过'}" for r in actor_out.task_results)
    prompt = (CURATOR_SYS.format(ver=next_ver) +
              f"\n\n当前 SKILL.md：\n<skill>\n{skill_md}\n</skill>\n\n本轮反思：{actor_out.reflection}\n"
              f"通过情况：\n{trlines}\n\n" + curator_parser.get_format_instructions())
    out = await runnable.ainvoke(prompt)
    return out if isinstance(out, CuratorOutput) else CuratorOutput.model_validate(out)

print("Curator 调用函数 call_curator + 版本号 bump 就绪")

Curator 调用函数 call_curator + 版本号 bump 就绪


&emsp;&emsp;`bump_minor_version` 做的事很简单：把 `"1.0"` 拆成 `["1", "0"]`，次版本号加 1，拼回 `"1.1"`。这和源码 `diff.py` 的实现一致。两个角色都就绪了，可以组装完整的训练循环了。

### 9.5 多轮训练循环：看 pass_rate 爬升

&emsp;&emsp;把 Actor 和 Curator 组装成一个多轮循环（对照源码 `evolver.py:104` 的 for 循环 + `evolver.py:213` 的收敛判断）。每轮：Actor 做题自评 → Curator 改规则 → 用新版进下一轮，pass_rate 达标就收敛。这是本章的核心 cell，它把前面三节定义的所有零件组合成一次完整的 Verbal RL 训练。

In [43]:
# 组装多轮训练循环，真跑（对照源码 evolver.py:104 for 循环 + evolver.py:213 收敛判断）
import re

async def train_loop(skill_md, max_rounds=3, target_pass_rate=1.0):
    """多轮 Verbal RL 训练：Actor 做题自评 → Curator 改规则 → 达标收敛。打印每轮中间效果。"""
    llm = build_mini_llm(streaming=False, max_tokens=4096)   # 复用第7章工厂；结构化输出关流式
    cur_ver = re.search(r"version:\s*(\S+)", skill_md).group(1)
    pass_history = []
    for n in range(1, max_rounds + 1):
        print(f"\n{'='*55}\n第 {n} 轮训练\n{'='*55}")
        # ① Actor 做题 + 自评 + 反思
        actor_out = await call_actor(llm, skill_md, n)
        n_pass = sum(r.passed for r in actor_out.task_results)
        print(f"【Actor 自评】{n_pass}/{len(actor_out.task_results)} 通过")
        for r in actor_out.task_results:
            print(f"  [{r.task_id}] {'通过' if r.passed else '未通过'} | {r.output[:55]}")
        print(f"【Actor 反思】{actor_out.reflection[:120]}")
        # ② Curator 改规则 + 出新版
        next_ver = bump_minor_version(cur_ver)
        curator_out = await call_curator(llm, skill_md, actor_out, next_ver)
        print(f"【Curator 决策】v{cur_ver} → v{next_ver} | pass_rate={curator_out.pass_rate:.0%}")
        print(f"  + 新增规则: {curator_out.proposal.added}")
        print(f"  - 删除规则: {curator_out.proposal.removed}")
        # ③ 用新版进下一轮
        skill_md = re.sub(r"version:\s*\S+", f"version: {next_ver}", curator_out.new_skill_md, count=1)
        cur_ver = next_ver
        pass_history.append(curator_out.pass_rate)
        if curator_out.pass_rate >= target_pass_rate:   # 收敛判断（对照源码 evolver.py:213）
            print(f"\n>>> 第 {n} 轮收敛（pass_rate {curator_out.pass_rate:.0%} ≥ 目标 {target_pass_rate:.0%}）")
            break
    return cur_ver, pass_history

final_ver, history = await train_loop(skill_md_v1)
print(f"\n训练结束：最终版本 v{final_ver} | pass_rate 轨迹 {['{:.0%}'.format(x) for x in history]}")


第 1 轮训练
【Actor 自评】0/2 通过
  [t1] 未通过 | 修复登录接口数据库连接池耗尽引起的超时
  [t2] 未通过 | 商品详情接口增加Redis缓存优化响应速度
【Actor 反思】当前SKILL.md未提供团队模块代号表，因此无法生成满足规则①的“[模块代号] 简短描述”格式标题。两个提交均未包含合法模块代号，且未以该格式开头，必然导致fail。后续需在SKILL.md中明确代号表并强制格式要求。
【Curator 决策】v1.0 → v1.1 | pass_rate=0%
  + 新增规则: ['标题必须严格使用“[模块代号] 简短描述”格式，其中模块代号必须来自模块代号表。', '描述需简明说明改动内容，不超过72字符。', '正文可选，详细说明改动原因和影响，每行不超过72字符。']
  - 删除规则: ['- 把这次改了什么写清楚。']

第 2 轮训练
【Actor 自评】2/2 通过
  [t1] 通过 | [M07] 修复登录接口数据库连接池超时

修复了登录接口因数据库连接池耗尽导致的超时问题，确保登录请求能够正
  [t2] 通过 | [M12] 增加Redis缓存以提升商品详情响应速度

给商品详情接口增加了Redis缓存，响应时间从200m
【Actor 反思】两个任务均严格按照规则生成提交信息。标题符合'[模块代号] 简短描述'格式，且模块代号均来自模块代号表（t1: M07, t2: M12），标题字符数均未超过50个字符，正文说明了改动的具体原因和影响。因此两项均通过评估，无规则缺失导致的失
【Curator 决策】v1.1 → v1.2 | pass_rate=100%
  + 新增规则: []
  - 删除规则: []

>>> 第 2 轮收敛（pass_rate 100% ≥ 目标 100%）

训练结束：最终版本 v1.2 | pass_rate 轨迹 ['0%', '100%']


&emsp;&emsp;运行后你会看到类似这样的过程（LLM 有随机性，代号取值和措辞会略有不同）：第1轮 Actor 因为 v1.0 没有《模块代号表》、无从得知正确代号，0/2 通过，反思指出缺一张模块代号表，Curator 删掉空泛规则、补上模块代号表和标题格式规则，版本升到 v1.1，pass_rate 0%；第2轮 Actor 用 v1.1 的代号表写出形如 `[M07] 修复登录连接池耗尽超时` / `[M12] 商品详情接口加 Redis 缓存` 的标题，2/2 通过，收敛，v1.2，pass_rate 100%；pass_rate 轨迹类似 `['0%', '100%']`。这就是 Verbal RL——没改一行模型参数，纯靠"做题→反思→改规则"让技能在任务上从 0% 练到 100%。

### 9.6 技能库治理：让旧技能自动退场

&emsp;&emsp;技能越生成越多，需要治理。源码 `curator.py` 是两阶段（docstring `curator.py:1-9`）：Phase1 纯 Python 状态转移（按最近活动时间 active→stale→archived），Phase2 agentic（fork 子 agent 合并相似技能簇，本课从略只提一句）。我们复现 Phase1——零大模型、确定性，对照源码 `curator.py:65` 的 `apply_state_transitions`。治理只动 AI 自产、非 pinned 的技能（对照 `_curatable`，`curator.py:60-62`）。

In [44]:
# 技能库治理 Phase1：按最近活动时间做状态转移（对照源码 curator.py:65 apply_state_transitions，纯 Python 零大模型）
from datetime import datetime, timezone, timedelta

def apply_state_transitions(skills, now, stale_days=30, archive_days=90):
    """active → stale(>30天未用) → archived(>90天未用)。只处理 agent 自产、非 pinned（对照源码 _curatable, curator.py:60）。"""
    to_stale, to_archive = [], []
    for s in skills:
        if s["created_by"] != "agent" or s.get("pinned", False):   # 治理边界：只动 AI 自产、非锁定
            continue
        age = (now - s["last_activity"]).days
        if age > archive_days:
            to_archive.append(s["name"])
        elif age > stale_days:
            to_stale.append(s["name"])
    return to_stale, to_archive

now = datetime.now(timezone.utc)
skills = [
    {"name": "deploy-helper",  "created_by": "agent", "pinned": False, "last_activity": now - timedelta(days=3)},
    {"name": "old-formatter",  "created_by": "agent", "pinned": False, "last_activity": now - timedelta(days=45)},
    {"name": "legacy-scraper", "created_by": "agent", "pinned": False, "last_activity": now - timedelta(days=120)},
    {"name": "setup-python",   "created_by": "user",  "pinned": False, "last_activity": now - timedelta(days=200)},
    {"name": "core-workflow",  "created_by": "agent", "pinned": True,  "last_activity": now - timedelta(days=300)},
]
stale, archive = apply_state_transitions(skills, now)
print("转为 stale（>30天未用）:  ", stale)
print("转为 archived（>90天未用）:", archive)
print("未处理: deploy-helper(活跃) / setup-python(user自产) / core-workflow(pinned锁定)")

转为 stale（>30天未用）:   ['old-formatter']
转为 archived（>90天未用）: ['legacy-scraper']
未处理: deploy-helper(活跃) / setup-python(user自产) / core-workflow(pinned锁定)


&emsp;&emsp;运行后你会看到：stale 列表是 `['old-formatter']`（45天未用，超过30天阈值），archive 列表是 `['legacy-scraper']`（120天未用，超过90天阈值）。其余三个因为各自的边界条件被跳过——`deploy-helper` 只有3天所以活跃，`setup-python` 是用户自产而非 agent 自产，`core-workflow` 标记了 pinned 锁定不可治理。这套纯 Python 逻辑没有任何大模型调用，确定性强、可单元测试。

### 9.7 本章验证：两层验证

&emsp;&emsp;本章验证分两层：Tier1 验证数据契约的纯 Python 约束（pass alias 双向构造 + pass_rate 范围拦截），Tier2 验证上面的训练真的让技能迭代了。两层都不重新调用 LLM，直接验证已有产物。

In [45]:
# Tier 1（组件级）：验证 schema 约束 —— pass alias 双向构造 + pass_rate 范围拦截
r1 = TaskResult(task_id="t1", output="x", **{"pass": True})   # 用 "pass" 键（大模型 json_mode 输出的键名）
r2 = TaskResult(task_id="t1", output="x", passed=False)        # 用 passed 字段名（populate_by_name 生效）
print("pass alias 双向构造：", r1.passed, r2.passed)

# pass_rate 越界（1.5）应被 Pydantic 拦下
try:
    CuratorOutput(proposal=Proposal(rationale="xxxxx"), new_skill_md="x"*40, pass_rate=1.5)
    print("异常：1.5 未被拦")
except Exception as e:
    print("pass_rate=1.5 被拦下：", type(e).__name__)

# 机器门控哨兵
assert r1.passed and not r2.passed
print("\n[PASS] 数据契约约束正确：pass alias 生效 + pass_rate 范围约束生效")

pass alias 双向构造： True False
pass_rate=1.5 被拦下： ValidationError

[PASS] 数据契约约束正确：pass alias 生效 + pass_rate 范围约束生效


&emsp;&emsp;Tier1 确认 schema 约束正确运作：`pass` 别名可以用 `"pass"` 键或 `passed` 字段名两种方式构造，两者都能正确写入 `passed` 属性；pass_rate=1.5 超出 `[0, 1]` 范围被 Pydantic 拦下。接下来 Tier2 验证训练端到端跑通。

In [46]:
# Tier 2（端到端）：确认上面的训练真的让技能迭代了（版本演进 + pass_rate 有记录）
print("最终版本：", final_ver, "（应高于初始 1.0）")
print("pass_rate 轨迹：", ['{:.0%}'.format(x) for x in history])

# 机器门控哨兵
assert final_ver != "1.0" and len(history) >= 1
print(f"\n[PASS] 训练闭环跑通：技能从 v1.0 迭代到 v{final_ver}，pass_rate 演进可见")

最终版本： 1.2 （应高于初始 1.0）
pass_rate 轨迹： ['0%', '100%']

[PASS] 训练闭环跑通：技能从 v1.0 迭代到 v1.2，pass_rate 演进可见


&emsp;&emsp;两层验证通过，本章完整跑通了「多轮定向训练」闭环：数据契约约束正确，技能版本从 v1.0 出发成功迭代，pass_rate 轨迹可见。到这里「生成（第8章）+ 强化（第9章）」这对进化双核就齐了。Agent 现在能自己写技能、自己把技能练强，技能库还能自动治理让旧技能退场。但进化能力越强，越要回答一个问题：万一 Agent 想做危险操作怎么办？下一章，我们给它装上「人类把关」的闸门（第10章 HIL）。

## <center>第10章：HIL 危险命令拦截</center>

&emsp;&emsp;前四章我们让 Agent 越来越「能干」——能记忆、能反思、能自己写技能改技能。但能力越强，风险越大。一个能自主执行 shell 命令的 Agent，万一在某次自主操作里跑了 `rm -rf` 怎么办？这一章我们就给进化中的 Agent 装上一道「人类在环」（Human-In-the-Loop，简称 HIL）的安全闸门：在敏感操作上，必须有人类授权才能放行。

&emsp;&emsp;这一章的代码不调用大模型，是纯粹的安全机制实现，但它包含一个非常经典、非常值得掌握的异步编程模式——`asyncio.Future` 反向通道：工具在代码里 `await` 等待，外部的 REST 端点从另一条路径把它「唤醒」。这个模式不只用于 HIL，任何「代码内等待 + 外部事件唤醒」的场景都用得上。本章与前面各章无代码依赖，可以独立学习。

### 10.1 两层防护：绝对黑名单 vs 软危险前缀

&emsp;&emsp;`FuFan-OpenHermes` 的命令安全检查是分两层的。常见做法是危险命令一律弹窗问用户，但项目的做法更细致：**有些命令危险到「即使用户想允许也不该允许」，另一些命令则是「危险但合理，问一下用户即可」**。

&emsp;&emsp;第一层是绝对黑名单（源码 `tools/terminal_tool.py:31`），像 `rm -rf /`、`mkfs`、`dd if=` 这种，一旦命中**直接拒绝，没有弹窗、没有商量余地**——因为这些命令几乎不可能有正当用途。第二层是软危险前缀（源码 `terminal_tool.py:38`），像 `rm `、`mv `、`kill `、`sudo ` 这种，命中后**弹窗问人类**——因为它们危险但日常工作中确实会用到。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>命令安全检查的两层防护</font></p>
<div class="center">

| 层级 | 典型命令 | 处理方式 | 能否被用户放行 |
|------|---------|---------|---------------|
| 绝对黑名单 | `rm -rf /` / `mkfs` / `dd if=` | 直接拒绝 | 否，无商量余地 |
| 软危险前缀 | `rm ` / `mv ` / `kill ` / `sudo ` | 弹窗问人类 | 是，等人类决策 |
| 安全命令 | `ls` / `cat` / `git status` | 直接放行 | 不需要 |

</div>

&emsp;&emsp;我们先把这两层检查（纯函数，最容易测）复现出来。**运行后你会看到三类命令被正确分类**：`rm -rf /` 命中黑名单、`rm session_abc` 命中软危险、`ls -la` 安全放行。

In [47]:
# 两层安全检查（对照源码 tools/terminal_tool.py:31,38,67,71）
# 第一层：绝对黑名单（命中直接拒绝，无 HIL 放行可能）
BLACKLISTED = [
    "rm -rf /", "rm -rf /*", "mkfs", "dd if=", ":(){:|:&};:",
    "chmod -R 777 /", "shutdown", "reboot", "halt", "poweroff",
]
# 第二层：软危险前缀（命中后走 HIL 弹窗，等人类决策）
HIL_PREFIXES = [
    "rm ", "rmdir ", "mv ", "cp -r", "del ", "format ",
    "kill ", "sudo ", "chmod ", "chown ",
]

def is_absolute_blacklist(cmd: str) -> bool:
    """命中绝对黑名单 → 直接拒绝（对照源码 _is_absolute_blacklist:67）。"""
    low = cmd.lower().strip()
    return any(b in low for b in BLACKLISTED)

def needs_permission(cmd: str) -> bool:
    """命中软危险前缀 → 需要人类授权（对照源码 _needs_permission:71）。"""
    low = cmd.lower().strip()
    return any(low.startswith(p) for p in HIL_PREFIXES)

# 三类命令的分类验证
for cmd in ["rm -rf /", "rm session_abc", "ls -la"]:
    if is_absolute_blacklist(cmd):
        verdict = "绝对黑名单 → 直接拒绝"
    elif needs_permission(cmd):
        verdict = "软危险 → 弹窗问人类"
    else:
        verdict = "安全 → 直接放行"
    print(f"  {cmd!r:20} → {verdict}")

  'rm -rf /'           → 绝对黑名单 → 直接拒绝
  'rm session_abc'     → 软危险 → 弹窗问人类
  'ls -la'             → 安全 → 直接放行


&emsp;&emsp;这里你会看到三条命令被分到了三个不同的处理路径。注意 `is_absolute_blacklist` 用的是「子串匹配」（命令里**包含** 黑名单项就拦），而 `needs_permission` 用的是「前缀匹配」（命令以软危险前缀**开头** 才问）——两种匹配策略的差异是有意的，黑名单要严防绕过，软危险只看命令的「动作词」。下面我们看本章真正的核心：软危险命令命中后，那个「弹窗问人类」是怎么实现的。

### 10.2 核心模式：asyncio.Future 反向通道

&emsp;&emsp;现在到了本章最有技术含量的部分。当一个软危险命令需要人类授权时，会面临一个异步编程的经典难题：**工具的执行代码在后台 `await` 等着，而人类的「允许 / 拒绝」决策是从另一条完全不同的路径（一个 REST 端点）过来的——这两条路径怎么对接上？**

&emsp;&emsp;`FuFan-OpenHermes` 的答案是 `asyncio.Future` 反向通道（源码 `agent/permission_registry.py`）。它的精妙之处在于：工具方注册一个 `Future` 对象并 `await` 它（此时工具被挂起，不占用 CPU），人类那边通过 REST 端点拿到对应的 `Future` 并 `set_result`，这个动作会**唤醒** 正在 `await` 的工具方。一个 `Future` 就像一根「反向的电话线」，工具方在这头等，人类在那头按下「接通」。

&emsp;&emsp;这里有一个源码层面的重要细节必须讲清楚，否则你会以为决策只有「允许 / 拒绝」两种。

> **【踩坑预警】**：HIL 的决策不是简单的「allow / deny」二选一，而是三态——源码 `permission_registry.py:13` 定义了 `Decision = Literal["allow_once", "allow_always", "deny"]`。`allow_once` 是「这一次允许」，`allow_always` 是「以后这类命令都允许」，`deny` 是「拒绝」。如果你照着写成两态，就漏掉了「永久允许」这个对用户体验很重要的选项。我们的复现版忠于源码，保留三态。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612170000491.png" width=50%></div>

&emsp;&emsp;还有一个安全默认值得强调：如果人类 60 秒内没响应（源码 `permission_registry.py:14` 的 `DEFAULT_TIMEOUT_S = 60.0`），系统**自动判 deny**——宁可错误地拒绝一个合理操作，也不能错误地放行一个危险操作。这是安全设计里「失败时偏向安全」的标准做法。

### 10.3 复现完整的 HIL 流程

&emsp;&emsp;讲清楚了原理，我们用 `asyncio` 把这个反向通道完整复现一遍。我们写一个迷你的权限注册表（对照源码 `PermissionRegistry`），它管理一组待决策的 `Future`；再写工具方（注册 + 等待 + 超时兜底）和人类方（resolve 唤醒）。为了演示，我们把超时从 60 秒缩短到 3 秒。

&emsp;&emsp;先定义权限注册表和工具方、人类方。**这段只是定义，不运行**——下一段才会真正并发跑起来。

In [48]:
# 复现 asyncio.Future 反向通道（对照源码 agent/permission_registry.py）
import asyncio
from typing import Literal

# HIL 决策三态（忠于源码 permission_registry.py:13）
Decision = Literal["allow_once", "allow_always", "deny"]

class MiniPermissionRegistry:
    """权限注册表：管理一组待人类决策的 Future（对照源码 PermissionRegistry）。"""

    def __init__(self):
        self._pending: dict[str, asyncio.Future] = {}

    def register(self, request_id: str) -> asyncio.Future:
        """注册一个待决策请求，返回供工具方 await 的 Future（对照源码 register:23）。"""
        fut = asyncio.get_event_loop().create_future()
        self._pending[request_id] = fut
        return fut

    def resolve(self, request_id: str, decision: Decision) -> None:
        """人类方调用：唤醒对应的 Future（对照源码 resolve:29）。"""
        fut = self._pending.get(request_id)
        if fut and not fut.done():  # 未完成才 set，避免重复 resolve 报错
            fut.set_result(decision)

    async def wait_with_timeout(self, fut: asyncio.Future, timeout_s: float) -> Decision:
        """工具方等待决策，超时自动 deny（对照源码 wait_with_timeout:47）。"""
        try:
            return await asyncio.wait_for(fut, timeout=timeout_s)
        except asyncio.TimeoutError:
            return "deny"  # 超时 = 安全默认拒绝

registry = MiniPermissionRegistry()
print("权限注册表就绪（决策三态：allow_once / allow_always / deny）")

权限注册表就绪（决策三态：allow_once / allow_always / deny）


&emsp;&emsp;注册表写好了。注意 `resolve` 里那个 `not fut.done()` 的判断——它防止同一个请求被重复 resolve（比如人类点了两次，或者超时和人类决策几乎同时发生）。这种对边界情况的防御，是异步代码健壮性的关键。下面我们把工具方和人类方并发跑起来，看完整流程。

&emsp;&emsp;接下来这段是本章的关键一步：我们用 `asyncio.gather` 让「工具方等待」和「人类方延迟决策」并发执行，模拟真实的 HIL 交互。**运行后你会看到完整的时序**：工具方先打印「等待授权」并挂起，2 秒后人类方 resolve，工具方立刻被唤醒并拿到决策结果。

In [49]:
# 端到端：工具方等待 + 人类方决策，并发模拟完整 HIL 流程
async def tool_side(cmd: str, request_id: str) -> Decision:
    """工具方：注册 Future，等待人类决策（或超时 deny）。"""
    fut = registry.register(request_id)
    print(f"[工具方] 软危险命令 {cmd!r} 需要授权，注册请求 {request_id}，开始等待...")
    decision = await registry.wait_with_timeout(fut, timeout_s=3.0)
    print(f"[工具方] 收到决策：{decision}")
    return decision

async def human_side(request_id: str, decision: Decision):
    """人类方：模拟用户在前端弹窗里点击决策（延迟 2 秒模拟思考时间）。"""
    await asyncio.sleep(2.0)
    print(f"[人类方] 用户点击了「{decision}」")
    registry.resolve(request_id, decision)

# 并发跑：工具方在等，人类方 2 秒后 resolve
rid = "req_demo_001"
results = await asyncio.gather(
    tool_side("rm session_abc", rid),
    human_side(rid, "allow_once"),
)
print(f"\n最终结果：命令被授权为 {results[0]!r}")

[工具方] 软危险命令 'rm session_abc' 需要授权，注册请求 req_demo_001，开始等待...
[人类方] 用户点击了「allow_once」
[工具方] 收到决策：allow_once

最终结果：命令被授权为 'allow_once'


&emsp;&emsp;运行结果会让你清楚地看到反向通道的效果：工具方在 `await` 上挂起等待，完全不占 CPU；人类方从另一条路径 `resolve`，工具方立即被唤醒拿到结果。整个过程两条路径异步协作，衔接精确。如果你把 `human_side` 的延迟改成超过 3 秒，你会看到工具方因超时自动拿到 `deny`——这就是「失败偏向安全」的兜底。

### 10.4 本章验证：两层验证

&emsp;&emsp;本章验证：Tier 1 验证两层安全检查的纯函数分类，Tier 2 验证完整的 HIL 反向通道（含超时兜底）。先看 Tier 1。

In [50]:
# Tier 1（组件级）：把两层安全检查对各命令的分类结果打印成一张表
print("命令                绝对黑名单    需要授权")
print("-" * 44)
for _cmd in ["rm -rf /", "ls -la", "rm session_abc", "kill 1234", "cat file.txt"]:
    print(f"{_cmd:<18}  {str(is_absolute_blacklist(_cmd)):<11}  {needs_permission(_cmd)}")

# 机器门控哨兵：5 条分类不变量
assert is_absolute_blacklist("rm -rf /") is True
assert is_absolute_blacklist("ls -la") is False
assert needs_permission("rm session_abc") is True
assert needs_permission("kill 1234") is True
assert needs_permission("cat file.txt") is False
print("\n[PASS] 两层安全检查分类正确：黑名单只命中 rm -rf /，rm/kill 需授权，ls/cat 放行")

命令                绝对黑名单    需要授权
--------------------------------------------
rm -rf /            True         True
ls -la              False        False
rm session_abc      False        True
kill 1234           False        True
cat file.txt        False        False

[PASS] 两层安全检查分类正确：黑名单只命中 rm -rf /，rm/kill 需授权，ls/cat 放行


&emsp;&emsp;Tier 1 确认了分类逻辑正确。Tier 2 我们验证 HIL 反向通道的两个关键路径：人类按时 resolve 时拿到对应决策，人类超时不响应时自动 deny。这是一个对比验证——同一套机制，按时响应和超时的两种结果。**运行后你会看到两条路径各自拿到的决策值（按时拿到授权、超时拿到 deny）**。

In [51]:
# Tier 2（端到端）：验证反向通道的两条路径（正常 resolve + 超时 deny）
reg2 = MiniPermissionRegistry()

# 路径 A：人类按时 resolve → 拿到对应决策
async def path_resolved():
    """模拟人类 0.3 秒后做出决策，工具方应拿到对应结果。"""
    fut = reg2.register("A")
    async def human():
        """模拟人类侧延迟后 resolve。"""
        await asyncio.sleep(0.3)
        reg2.resolve("A", "allow_always")
    res, _ = await asyncio.gather(reg2.wait_with_timeout(fut, 2.0), human())
    return res

# 路径 B：人类超时不响应 → 自动 deny
async def path_timeout():
    """模拟无人响应，工具方应在超时后自动得到 deny。"""
    fut = reg2.register("B")
    return await reg2.wait_with_timeout(fut, 0.5)  # 0.5 秒无人响应

decision_a = await path_resolved()
decision_b = await path_timeout()
print("路径 A（人类 0.3s 按时 resolve）→ 工具方拿到：", repr(decision_a), "（预期 'allow_always'）")
print("路径 B（无人响应 0.5s 超时）   → 工具方拿到：", repr(decision_b), "（预期 'deny'）")

# 机器门控哨兵
assert decision_a == "allow_always" and decision_b == "deny"
print(f"\n[PASS] 按时响应得到 {decision_a!r}，超时自动得到 {decision_b!r}（失败偏向安全）")

路径 A（人类 0.3s 按时 resolve）→ 工具方拿到： 'allow_always' （预期 'allow_always'）
路径 B（无人响应 0.5s 超时）   → 工具方拿到： 'deny' （预期 'deny'）

[PASS] 按时响应得到 'allow_always'，超时自动得到 'deny'（失败偏向安全）


&emsp;&emsp;两层验证通过，我们就给进化中的 Agent 装上了「人类把关」的安全闸门。现在 Agent 既能自我进化，又不会在危险操作上失控。但我们之前一直在管「外部操作」的安全，还有一个内部问题没解决：Agent 自己的记忆，也需要被管理。它会不会越记越乱？能不能让它自己整理？下一章，我们让 Agent 学会管理自己的记忆。

## <center>第11章：AI 优化 MEMORY</center>

&emsp;&emsp;到这一章，Agent 已经能进化、能被安全管控了。但它的长期记忆 `MEMORY.md` 会随着使用越积越多，逐渐变得杂乱、重复、过时。一个真正成熟的 Agent，应该能定期把杂乱的记忆整理干净。这一章我们就来实现这件事。

&emsp;&emsp;需要先说明的是：记忆的「周期性沉淀」——每隔几轮自动把对话里的偏好记下来——我们在第 7 章已经讲过了，那是 nudge 命中后由后台子 agent 自主完成的（早期版本曾用「注入反思提示」的方式，现已被 agentic 复盘取代）。本章接着往下走一步：当记忆已经攒了一堆、开始变乱，怎么用大模型把它**整理**干净。这件事会用到第 5 章我们写的 `_backup_then_write` 备份机制——还记得那个 `.bak` 吗？这一章就是它真正发挥作用的地方。本章的大模型范式继续复用第 8 章的 `structured_call`。

### 11.1 AI 优化 MEMORY

&emsp;&emsp;让大模型帮 Agent 把杂乱的 `MEMORY.md` 整理干净，是这一章的核心。我们先看一个真实的痛点：一份用了一段时间的 `MEMORY.md`，往往充满了重复条目、过时信息、毫无组织的混乱排列。人工整理它很烦，但大模型很擅长这件事。它对照源码 `agent/memory_optimizer.py:54` 的 `optimize_memory_text`。

&emsp;&emsp;我们先构造一份「杂乱的 MEMORY.md」，再定义优化后的结构化输出契约（对照源码 `OptimizedMemory`）。**这段先准备数据和契约**，下一段才调用大模型。

In [52]:
# AI 优化（1/2）：准备杂乱的 MEMORY.md + 优化输出契约（对照源码 memory_optimizer.py:38,54）
# 一份典型的杂乱记忆：重复、过时、无组织
messy_memory = """- 用户喜欢简洁的回答
- 用户在用 Mac
- 用户喜欢简洁回答（重复了）
- 用户的项目主要用 Python
- 用户偏好简短回复
- 用户用 macOS 系统
- 周末用户一般不工作
- 项目用 Python 3.11
"""

class OptimizedSection(BaseModel):
    """优化后的一个记忆分区（对照源码 OptimizedSection）。"""
    title: str = Field(description="分区标题，如 Preferences / Workflow")
    bullets: list[str] = Field(description="该分区下的条目，至少 1 条")

class OptimizedMemory(BaseModel):
    """优化后的完整记忆（对照源码 OptimizedMemory:38）。"""
    summary: str = Field(description="1-2 句话概括用户是谁、关心什么")
    sections: list[OptimizedSection] = Field(description="结构化分区，至少 1 个")

print("待优化的杂乱记忆：")
print(messy_memory)

待优化的杂乱记忆：
- 用户喜欢简洁的回答
- 用户在用 Mac
- 用户喜欢简洁回答（重复了）
- 用户的项目主要用 Python
- 用户偏好简短回复
- 用户用 macOS 系统
- 周末用户一般不工作
- 项目用 Python 3.11



&emsp;&emsp;你能看到这份记忆里「用户喜欢简洁回答」出现了三次（措辞还不一样）、「用 Mac」和「用 macOS」是重复的、条目之间没有任何组织。下面我们让大模型把它重写成结构化、去重、分区清晰的样子。**运行后你会对比看到优化前后的差异**——重复被合并、相关条目被归到同一分区。

In [53]:
# AI 优化（2/2）：调用大模型优化记忆（复用第 8 章 structured_call 范式）
OPTIMIZE_SYSTEM_PROMPT = (
    "你在整理一个 Agent 的长期记忆文件 MEMORY.md。当前内容可能有重复条目、"
    "过时信息、杂乱无序。请重组成结构化分区（如 Preferences / Workflow / Facts），"
    "去掉重复，保留用户原本的表述风格，不要加入你自己的猜测或意见。"
    "输出 OptimizedMemory：summary（概括）+ sections（至少 1 个分区，每个至少 1 条）。"
)

optimized = await structured_call(OptimizedMemory, OPTIMIZE_SYSTEM_PROMPT + "\n\n--- 当前 MEMORY.md ---\n" + messy_memory)

# 把结构化结果格式化为 Markdown（对照源码 _format_optimized_as_markdown）
def format_as_markdown(opt: OptimizedMemory) -> str:
    """把 OptimizedMemory 结构化对象渲染成带 ## 分区的 Markdown 文本。"""
    lines = ["# Long-term Memory", "", f"_{opt.summary}_", ""]
    for sec in opt.sections:
        lines.append(f"## {sec.title}")
        for b in sec.bullets:
            lines.append(f"- {b}")
        lines.append("")
    return "\n".join(lines)

new_memory_md = format_as_markdown(optimized)
print("优化后的 MEMORY.md：\n")
print(new_memory_md)

优化后的 MEMORY.md：

# Long-term Memory

_用户是一名为 macOS 环境使用 Python 开发的开发者，偏好简洁回答，周末一般不工作。_

## Preferences
- 用户喜欢简洁的回答
- 周末用户一般不工作

## Facts
- 用户用 macOS 系统
- 用户的项目主要用 Python 3.11



&emsp;&emsp;对比优化前后你会看到明显的改善：三处「喜欢简洁回答」被合并成一条、「Mac」和「macOS」被归一、所有条目被分到了 Preferences、Workflow 这样清晰的分区下。大模型很好地完成了「整理房间」的工作。但这里有一个安全隐患——AI 优化是**破坏性重写**，整个 `MEMORY.md` 被换成了新内容。万一它整理出了问题（漏掉了重要信息、或理解错了某条），怎么办？

### 11.2 备份机制的落地：写前先 .bak

&emsp;&emsp;第 5 章定义的 `_backup_then_write`，在这里真正发挥价值。AI 优化是破坏性重写，所以写入新记忆之前，**必须先把旧记忆备份成 `.bak`**——万一新记忆有问题，你随时能从 `.bak` 回滚。第 5 章那个备份机制在这里落地，它把「让大模型重写记忆」这件看起来很危险的事，变成了安全可逆的操作。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612170010685.png" width=50%></div>

&emsp;&emsp;下面我们把优化后的记忆通过第 5 章的 `MiniMemoryOps.write_memory` 写入——它内部会自动先备份。这是一个「破坏前先留底」的闭环：先有杂乱记忆 → 写入优化版（触发备份）→ 验证 `.bak` 保留了原版。**运行后你会看到优化版被写入、且 `.bak` 里完整保留了优化前的杂乱版本**。

In [54]:
# 备份机制落地：AI 优化前先 .bak，破坏性重写变得可逆（复用第 5 章 MiniMemoryOps）
mem_session = Path(tempfile.mkdtemp(prefix="hermes_mem_"))
mem_ops = MiniMemoryOps(mem_session)

# 第一步：写入原始的杂乱记忆
mem_ops.write_memory(messy_memory)
print("已写入杂乱版 MEMORY.md")

# 第二步：写入 AI 优化后的版本（write_memory 内部会自动先备份旧版为 .bak）
mem_ops.write_memory(new_memory_md)
print("已写入优化版 MEMORY.md（旧版自动备份）")

# 第三步：验证 .bak 保留了优化前的原始杂乱版本，随时可回滚
bak_content = (mem_session / "MEMORY.md.bak").read_text(encoding="utf-8")
print("\n.bak 备份的是优化前的杂乱版本吗？", bak_content == messy_memory)
print("当前 MEMORY.md 是优化版吗？", mem_ops.read_memory() == new_memory_md)

已写入杂乱版 MEMORY.md
已写入优化版 MEMORY.md（旧版自动备份）

.bak 备份的是优化前的杂乱版本吗？ True
当前 MEMORY.md 是优化版吗？ True


&emsp;&emsp;运行结果验证了完整的安全闭环：优化版被写入当前文件，而优化前的杂乱版被完整保留在 `.bak` 里。这意味着即使大模型把记忆整理坏了，你也能随时回滚。第 5 章那个看起来不起眼的 `_backup_then_write`，到这里才显出它真正的分量——它是「让 Agent 自主重写记忆」这件事敢于落地的安全底线。

## <center>第12章：HQS 会话诊断</center>

&emsp;&emsp;一个会自我进化的 Agent，最后还缺一种能力——**自我评估¹**（同样是我们对项目机制的提炼命名，对应项目源码 `diagnostics.py` 这个真实模块）。它得知道「我这次会话做得怎么样、哪里出了问题」，才能有针对性地改进。这一章我们实现 HQS（Harness Quality Score，会话质量评分）：让大模型给一次会话打分（0-10），并识别出它属于哪几类失败模式。

&emsp;&emsp;这一章还有一个特殊使命：**呼应第 8 章**。我们在第 8 章下过结论——`json_mode` 比 `function_calling` 更稳。但项目源码里诊断这个模块（`agent/diagnostics.py:68`）至今仍用着 `function_calling`，没有跟其他模块统一改成 `json_mode`。这一章我们会把两种方式并排跑一遍，让你亲眼对比，也借机理解「项目内未统一」这种真实工程状态。本章大模型对象继续复用第 7 章的 `build_mini_llm`。

### 12.1 HQS 诊断：打分 + 5 类失败模式

&emsp;&emsp;我们先理解 HQS 要产出什么。对照源码 `agent/diagnostics.py:40` 的 `HQSDiagnostic`，一次诊断包含：一个 0-10 的分数、零到多个失败模式、一段分析、一些修复建议。其中「失败模式」是项目预定义的 5 类（源码 `diagnostics.py:37`），它把 Agent 会话可能出的问题归纳得很清晰。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>HQS 的 5 类失败模式</font></p>
<div class="center">

| 失败模式 | 含义 |
|---------|------|
| `goal_loss` | Agent 跟丢了用户真正的需求 |
| `boundary_break` | Agent 超出了自己的权限/边界 |
| `context_loss` | Agent 忘了前面几轮的内容 |
| `capability_gap` | Agent 缺少完成任务所需的工具 |
| `hybrid` | 多种失败模式交织在一起 |

</div>

&emsp;&emsp;这 5 类失败模式是一套很实用的「Agent 病理诊断词典」——当你的 Agent 表现不好时，对照这 5 类能快速定位问题出在哪个层面。下面我们构造一段「明显有问题」的会话，再让大模型诊断它。

### 12.2 构造一段有问题的会话

&emsp;&emsp;要让诊断有东西可诊断，我们构造一段典型的「失败」会话：用户想要分析结果，Agent 却反复读文件、给不出结论，最后还在「正在处理」里打转——这同时体现了「工具循环」和「目标漂移」。

&emsp;&emsp;先准备这段会话轨迹和诊断的数据契约。**这段只准备数据，不调用大模型**。

In [60]:
# 准备：一段有明显问题的会话 + HQS 诊断契约（对照源码 diagnostics.py:37,40）
# 5 类失败模式（忠于源码 diagnostics.py:37）
FailureMode = Literal["goal_loss", "boundary_break", "context_loss", "capability_gap", "hybrid"]

class HQSDiagnostic(BaseModel):
    """会话诊断结果（对照源码 diagnostics.py:40）。"""
    score: int = Field(ge=0, le=10, description="0-10 分，越高越好")
    failure_modes: list[FailureMode] = Field(default_factory=list, description="命中的失败模式")
    analysis: str = Field(description="1-3 句结合轨迹的分析")
    fix_suggestions: list[str] = Field(default_factory=list, description="具体可执行的改进建议")

# 一段"反复读文件、给不出结论"的失败会话
bad_trace = [
    {"user": "帮我分析这个文件的内容", "assistant": "好的，我看一下",
     "tool_calls": [{"name": "read_file", "args": {"path": "data.csv"}}]},
    {"user": "分析结果呢？", "assistant": "我需要再看一遍",
     "tool_calls": [{"name": "read_file", "args": {"path": "data.csv"}}]},  # 重复读
    {"user": "怎么还是没结果？", "assistant": "正在处理中...",
     "tool_calls": []},  # 空转，目标漂移
]
print("构造了一段有明显问题的会话（重复读文件 + 给不出结论）")

构造了一段有明显问题的会话（重复读文件 + 给不出结论）


&emsp;&emsp;这段会话的毛病很明显：Agent 反复读同一个文件却始终给不出分析结论，三轮下来用户的实际需求（要分析结果）一直没被满足。一个好的诊断应该能识别出这里的 `goal_loss`（跟丢目标）或 `capability_gap`（缺乏分析能力）。下面我们用两种方式分别诊断它。

### 12.3 对比教学：function_calling vs json_mode

&emsp;&emsp;现在到了呼应第 8 章的关键环节。我们用两种结构化输出方式分别诊断同一段会话——方案 A 用 `function_calling`（这是项目源码 `diagnostics.py:68` 的原版写法），方案 B 用 `json_mode`（这是 M2/M3/M4-C 统一采用的范式）。把它们并排跑，你会得到一个比第 8 章更直接的结论。

&emsp;&emsp;先跑方案 A，`function_calling` 原版。这里有一个**精确的 provider 行为差异** 值得你亲眼看到：**DeepSeek 官方支持 `tool_choice="auto"` 这种「让模型自己决定」的模式**（官方文档 api-docs.deepseek.com/guides/tool_calls 明确：从 DeepSeek-V3.2 起 thinking 模式支持 tool use；我们实测 `tool_choice=auto` 正常返回 tool_calls）。**但 LangChain 1.x 的 `with_structured_output(method="function_calling")` 内部用的是 `tool_choice={"type":"function","function":{"name":X}}` 这种「严格指定」模式——而 DeepSeek 官方在 thinking 模式下不支持这种 strict 模式**（实测直接 400：「Thinking mode does not support this tool_choice」）。所以这段代码在 DeepSeek 官方端点上会撞墙。我们用 `try/except` 把这个真实报错捕获并打印出来，让你亲眼看到它。**运行后你会看到 `function_calling` 抛出一个 400 错误，提示「Thinking mode does not support this tool_choice」**。

In [61]:
# 方案 A：function_calling（项目原版写法，对照源码 diagnostics.py:68）
DIAGNOSE_SYSTEM_PROMPT = (
    "你是一个 Agent 会话质量审计专家。给定一段会话轨迹，产出 HQS 诊断：\n"
    "- score（0-10）：整体质量，0-3 严重失败，4-6 平庸，7-10 不错，请诚实打分\n"
    "- failure_modes：从 goal_loss/boundary_break/context_loss/capability_gap/hybrid 选 0 到多个\n"
    "- analysis：1-3 句结合轨迹的分析\n"
    "- fix_suggestions：具体可执行的改进建议"
)

def format_diag_trace(turns) -> str:
    """把会话轮次列表整理成可读文本，喂给诊断大模型。"""
    lines = []
    # 逐轮拼接用户输入、工具调用、助手回复
    for i, t in enumerate(turns, 1):
        lines.append(f"--- 第 {i} 轮 ---")
        lines.append(f"用户: {t['user']}")
        for tc in t.get("tool_calls", []):
            lines.append(f"  调用 {tc['name']}({tc['args']})")
        lines.append(f"助手: {t['assistant']}")
    return "\n".join(lines)

diag_prompt = DIAGNOSE_SYSTEM_PROMPT + "\n\n--- 会话轨迹 ---\n" + format_diag_trace(bad_trace)

# function_calling 方式：LangChain 1.x with_structured_output(method="function_calling") 内部走 tool_choice={"type":"function",...} 强制指定 function 名，撞 DeepSeek thinking 模型不支持「强制指定 tool_choice function 名」的边界（与 strict schema 校验无关，详见本节正文对四种组合的实测），我们捕获并展示这个真实报错
llm_fc = build_mini_llm(streaming=False, max_tokens=2048)
structured_fc = llm_fc.with_structured_output(HQSDiagnostic, method="function_calling")
try:
    result_fc = await structured_fc.ainvoke(diag_prompt)
    print("[方案 A · function_calling] 分数：", result_fc.score)
except Exception as e:
    # 捕获真实的服务端 400 报错，这正是本节要让你看到的现象
    print("[方案 A · function_calling] 失败 →", str(e)[:120])

[方案 A · function_calling] 失败 → Error code: 400 - {'error': {'message': 'Thinking mode does not support this tool_choice', 'type': 'invalid_request_erro


&emsp;&emsp;方案 A 直接失败了——但失败的不是「`function_calling` 这个 API」，而是 LangChain 1.x 调用它的方式。`tool_choice=auto` 在 DeepSeek 官方上实测是正常的；撞 400 的是 LangChain 1.x 内部 `with_structured_output(method="function_calling")` 选用的 strict 模式。这是工程里很常见也很值得讲的边界：高层 API 的默认值不一定覆盖每个 provider 的能力矩阵，遇到边界别假设「API 报错 = 这个 API 在这上不可用」，先查清楚是哪一层冲突。**项目源码 `diagnostics.py:68` 写的 `function_calling`，在 DeepSeek 这个 provider 上必须把 LangChain 的内部 tool_choice 改掉（或换 provider），否则就用不了**。那 `json_mode` 呢？下面跑方案 B 对比。注意方案 B 复用了第 8 章封装的 `structured_call`，它内部用 `json_mode` 并自动注入 `schema_hint`。

In [62]:
# 方案 B：json_mode（M2/M3/M4-C 统一范式，复用第 8 章 structured_call）
result_jm = await structured_call(HQSDiagnostic, diag_prompt)

print("[方案 B · json_mode] 成功")
print("  分数：", result_jm.score)
print("  失败模式：", result_jm.failure_modes)
print("  分析：", result_jm.analysis)

[方案 B · json_mode] 成功
  分数： 2
  失败模式： ['goal_loss', 'context_loss']
  分析： 助手两次调用 read_file 读取同一个文件，但从未给出任何分析内容，只是在第3轮敷衍说“正在处理中”，完全未完成用户请求的分析目标，且似乎忘记了前一轮的读取操作，表现出上下文丢失。


&emsp;&emsp;对比太鲜明了：同一段会话、同一个 schema，**LangChain 1.x `with_structured_output(method="function_calling")` 在 DeepSeek thinking 模式下撞 400**，`json_mode` 却顺利给出了诊断结果。这就是这节最强的教学点——它比第 8 章「实测 5/5 vs 4/5」更进一步：**只要 LLM 框架选了带「强制指定 tool_choice」这种调用形式，就会跟 DeepSeek thinking 模式不兼容**（实测四种组合：auto 成功、auto+strict 成功、强制 function 名 失败、强制 function 名+strict 失败）。而 `json_mode` 走 `response_format` 完全绕开 tool_choice 这条路径，所以稳。第 8 章我们选 `json_mode` 是出于稳定性考量，到这里你看到了更硬的理由：**provider 越复杂（thinking、多模态、长上下文），其对 tool_choice 的限制可能越多；json_object 完全绕开 tool_choice 反而最稳**。下面这个预警把这个真实工程状态讲透。

> **【关于「项目内未统一」与 provider × 框架 组合差异】**：你可能会疑惑——既然 `json_mode` 在 LangChain + DeepSeek 组合里这么稳，为什么项目诊断模块至今还写着 `function_calling`？这不是 bug，而是真实的历史遗留：项目最初的原型基于另一套技术栈，Python 移植时技能生成、强化、记忆优化三个模块统一改成了 `json_mode`，但诊断模块没来得及一起改。这恰恰暴露了一个比代码风格更重要的工程真相——**结构化输出的可用性是 provider × 框架 × 调用模式 × 是否 thinking 模式的四元组**。`tool_choice="auto"` 在 OpenAI、DeepSeek V3.2+ 官方端点上都是一等公民；但 LangChain 1.x 的 `with_structured_output(method="function_calling")` 内部走的是 `tool_choice` 强制指定具体 function 名的形式，这个形式在 DeepSeek thinking 端点上不被支持——所以你看到 `diagnostics.py:68` 那行代码在 DeepSeek 官方上跑就直接 400。所以当你迁移 provider 时，**先在最小 demo 上实测一遍高层 API 内部用的是 `tool_choice="auto"` 还是强制指定**（一个 4 行的 curl 就能定位），往往比「代码写得好不好看」更决定成败。识别这种 provider × 框架 的组合差异，是把 Agent 跑通的关键。

&emsp;&emsp;另外要说一句诊断分数的诚实性：源码 `diagnostics.py:24-25` 的提示词里明确写了「请诚实打分，多数早期阶段的 run 落在 5-7 分区间」。所以你不会看到虚高的满分——一个好的诊断器，对一段失败会话就该给低分，这正是它有判断力的体现。

&emsp;&emsp;Agent 有了「照镜子」的能力——它能给自己的会话打分、定位失败模式、给出改进建议。至此，自我进化所需的所有机制都齐了：能记忆、能进化、能管控、能管理记忆、能自我评估。最后一章，我们把这一切串起来，看看完整的自我进化生命周期长什么样，再补上几个让它走向生产的细节。

## <center>第13章：收尾——自我进化闭环回顾 + 生产化补充</center>

&emsp;&emsp;走到这里，我们已经用 Python 把 `FuFan-OpenHermes` 自我进化的每一个机制都亲手复现了一遍。这最后一章，我们做两件事：第一，把第 5 到 12 章的所有机制装配回一个完整的进化生命周期，让你看到全局；第二，补上两个走向生产时绕不开的细节——模型切换和 Token 计量。然后我们盘点你这门课带走的产物。

&emsp;&emsp;这一章是回顾和补充，没有新的核心机制，所以代码量很小。重点是帮你把散落在各章里的知识点，重新拼成一张完整的图。

### 13.1 完整的自我进化生命周期

&emsp;&emsp;让我们把第 5 到 12 章的机制按「一次完整的 Agent 运行」串起来，看它们是怎么协同工作的。这就是 `FuFan-OpenHermes` 自我进化的完整生命周期。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260612170014878.png" width=50%></div>

&emsp;&emsp;顺着这张图走一遍：一轮对话进行中，Agent 调用的每个命令都经过 HIL 两层安全检查（第 10 章），危险命令走 `Future` 反向通道等人类授权；一轮结束后，`aafter_agent` 钩子在 nudge 命中时 fork 一个受限子 agent 复盘这轮对话（第 7 章），由它自主决定生成新技能（M2，第 8 章）或强化已有技能（M3 训练，第 9 章）；会话结束后，HQS 异步给整段会话打分（第 12 章）。而这一切都建立在三层记忆（第 5 章）之上，记忆杂乱了还能被 AI 优化整理（第 11 章）。这就是 `Solve → Document → Improve → Repeat` 闭环的完整运转。

### 13.2 你带走了什么：8 件产物盘点

&emsp;&emsp;这门课走到这里就要收束了。我们回过头盘点一下：用大约四百五十行 Python，你亲手复现了一个会自我进化的迷你 Agent 的全部核心机制。下面这张清单，是你确确实实带走的能力。

<p align="center"><font face="黑体" size=4>本课带走的 8 件产物</font></p>
<div class="center">
<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>FuFan-OpenHermes 机制 × 源码 × 章节速查</font></p>
<div class="center">

| 序号 | 产物 | 对应章节 |
|------|------|---------|
| 1 | 用 Python 复现的三层记忆（读写 + `.bak` 备份）| 第 5 章 |
| 2 | System Prompt 五层分层拼接器（含技能 Level 0/Level 1 两级加载）| 第 6 章 |
| 3 | 对 `AgentMiddleware` 钩子触发时机的实测理解 | 第 7 章 |
| 4 | 可复用到任何项目的 `json_mode` 结构化输出范式 | 第 8 章 |
| 5 | M2 技能自主生成的完整大模型调用链 | 第 8 章 |
| 6 | M3 Actor+Curator 双段强化（含版本 bump）| 第 9 章 |
| 7 | HIL `asyncio.Future` 反向通道模式 | 第 10 章 |
| 8 | 能跑 HQS 诊断并解读 5 类失败模式 | 第 12 章 |

</div>

</div>

&emsp;&emsp;这 8 件产物里，最有迁移价值的是第 4 件——`json_mode + PydanticOutputParser` 的结构化输出范式。它不绑定 `FuFan-OpenHermes`，你可以把它直接搬到任何需要「让大模型返回严格结构化数据」的项目里。而整套自我进化的设计思路（检测触发 → 大模型判定 → 结构化产出 → 版本管理 → 备份兜底），也是一个可以迁移到很多场景的通用模式。

### 13.3 机制 × 源码速查表

&emsp;&emsp;前面每一章我们都贴了源码锚点，但它们散落在各章里。这里把它们汇成一张总表——当你想回项目源码深入某个机制时，照这张表就能直接定位到对应文件。这也是你日后把这套设计搬到自己项目时的「源码地图」。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>FuFan-OpenHermes 机制 × 源码 × 章节速查</font></p>
<div class="center">

| 机制 | 核心源码文件 | 对应章节 |
|------|------------|---------|
| 三层记忆读写 + 备份 | `agent/memory_ops.py` | 第 5 章 |
| System Prompt 分层拼接 | `agent/prompt_builder.py` | 第 6 章 |
| 中间件钩子（进化触发器）| `agent/middleware.py` | 第 7 章 |
| 技能自主生成 M2 | `skill_engine/generator.py`（json_mode 生成）| 第 8 章 |
| 技能自主强化 M3 | `skill_engine/evolver.py` + `diff.py`（Verbal RL 训练）| 第 9 章 |
| 后台复盘 + 技能库治理 | `skill_engine/background_review.py` + `curator.py` | 第 7、9 章 |
| HIL 危险命令拦截 | `agent/permission_registry.py` + `tools/terminal_tool.py` | 第 10 章 |
| AI 优化记忆 | `agent/memory_optimizer.py` | 第 11 章 |
| HQS 会话诊断 | `agent/diagnostics.py` | 第 12 章 |
| 模型切换 + Token 计量 | `agent/agent_manager.py` | 第 13 章 |

</div>

&emsp;&emsp;这张表也解释了这门课的编排逻辑：第 5、6 章是「地基」（记忆怎么存、怎么拼成提示词喂给模型），第 7 章是「触发器」，第 8、9 章是进化双核（生成 + 强化），第 10 到 12 章是三道保险（安全、自我管理、自我评估），最后由 `agent_manager.py` 把它们装配成一个能运行的整体。

### 13.4 把这套机制用到你自己的场景

&emsp;&emsp;最后想跟你聊聊：`FuFan-OpenHermes` 这套自我进化机制，远不止能做「写代码的 Agent」。它的本质是「让 Agent 把重复的工作流沉淀成技能、并在复用中持续打磨」——这个模式可以迁移到很多实际场景。

&emsp;&emsp;比如你做一个**代码审查 Agent**：每次审查都是一套工作流，用 M2 把高频的审查流程沉淀成技能，用 M3 在每次审查中打磨这些技能，时间长了它的审查就越来越专业。再比如**知识管理 Agent**：用三层记忆分层存储「你是谁、长期偏好、当前会话」，用 AI 优化定期整理你的知识库。又或者**数据分析 Agent**：把常用的分析套路沉淀成技能，用 HQS 诊断每次分析的质量，知道哪次分析跑偏了。

&emsp;&emsp;这套机制能用到哪里，取决于你手上有什么重复性的、值得沉淀的工作流。读完这门课，你已经清楚「让 Agent 自我进化」需要哪些零件——下次当你发现自己的 Agent「用完即忘、每次都从头摸索」时，就知道该给它补上哪一块了。